In [35]:
import os
import re
import pandas as pd
import json
from bs4 import BeautifulSoup
from datetime import datetime

In [36]:
def transform_time(df, col="time"):
    format = '%Y-%m-%d %H:%M:%S.%f'
    if col == 'time':
        df['time'] = df['time'].apply(lambda x: datetime.strptime(x, format))
    else:
        df[col] = df[col].apply(lambda x: datetime.strptime(x, format))
    return df

def second_difference(d2, d1):
    format = '%Y-%m-%d %H:%M:%S.%f'
    d2 = datetime.strptime(d2, format)
    d1 = datetime.strptime(d1, format)
    return (d2 - d1).seconds

def minutes_difference(d2, d1):
    return second_difference(d2, d1) / 60

In [37]:
participants_groups = {
    "1": {"users": [1379, 1380], "user_ids": ['67ce982f285fac3f07bcf5e7', '67ce9cb4285fac3f07bcf5ec'], "documents": ["67ce983b285fac3f07bcf5e8"]},
    "2": {"users": [1376, 1377], "user_ids": ['67cb6627285fac3f07bcf59b', '67cb6623285fac3f07bcf59a'], "documents": ["67c97527285fac3f07bcf587"]},
    "3": {"users": [1378, 1374], "user_ids": ['67cc9f69285fac3f07bcf5c8', '67cdfa30285fac3f07bcf5cc'], "documents": ["67c88f88285fac3f07bcf585"]},
    "4": {"users": [1383, 1384, 1385], "user_ids": ['67bc761a7a670b68746905a0', '67d444dd0b9539e69a3a86bb', '67bc43607a670b6874690577'], "documents": ["67d43b390b9539e69a3a86b9"]},
    "5": {"users": [1387, 1390], "user_ids": ['67d5d8120b9539e69a3a86d7', '67e18e95edfe680c1b0f169f'], "documents": ["67d5d8160b9539e69a3a86d8"]},
    "6": {"users": [1388, 1389], "user_ids": ['67d82dc4edfe680c1b0f1642', '67d82e0aedfe680c1b0f1643'], "documents": ["67d08e87285fac3f07bcf64b"]},
    "7": {"users": [2456, 2460], "user_ids": ['68835b0587cf5fec60dc8020', '6887b97087cf5fec60dc8060'], "documents": ["68833fd687cf5fec60dc801d"]},
    "8": {"users": [2453, 2455], "user_ids": ['687de5d187cf5fec60dc7fef', '6880e0f187cf5fec60dc7ffe'], "documents": ["6879e8fd87cf5fec60dc7fe1"]},
    "9": {"users": [2461, 2464, 2457], "user_ids": ['6888eec887cf5fec60dc8085', '688a52ad87cf5fec60dc8096', '688b1c7887cf5fec60dc80b0'], "documents": ["688364bf87cf5fec60dc8026"]},
    "10": {"users": [2459, 2465], "user_ids": ['6888bbce87cf5fec60dc8072', '688bdb6387cf5fec60dc80d0'], "documents": ["6877c26e87cf5fec60dc7fda"]},
    "11": {"users": [2463, 2475], "user_ids": ['6889e71f87cf5fec60dc8094', '68a75365ecc07d2944852275'], "documents": ["6889e57087cf5fec60dc8092"]},
    "12": {"users": [2468, 2469], "user_ids": ['6898407be90ffc190a96c287', '689c3c25e90ffc190a96c28b'], "documents": ["688c6add87cf5fec60dc80ef"]},
    "13": {"users": [2467, 2476], "user_ids": ['68a5a0bde90ffc190a96c2ff', '68a59766e90ffc190a96c2f4'], "documents": ["68907a6e87cf5fec60dc8107"]},
    "14": {"users": [2470, 2471], "user_ids": ['689e2181e90ffc190a96c2df', '689e22d5e90ffc190a96c2e0'], "documents": ["6896071ee90ffc190a96c285"]},
} 


participants_id = [1378, 1374, 1376, 1377, 1379, 1384, 1383, 1385, 1387, 1388, 1389, 1390, 1380]
participants_id_wo_directusers = [1378, 1374, 1376, 1377, 1379, 1387, 1388, 1389, 1390, 1380] #removed 1383 1384 1385
participants_id_study_extended = [2456, 2460, 2453, 2455, 2461, 2464, 2457, 2459, 2465, 2463, 2475, 2468, 2469, 2467, 2476, 2470, 2471]

# join lists
participants_id = participants_id + participants_id_study_extended
participants_id_wo_directusers = participants_id_wo_directusers + participants_id_study_extended

In [38]:
finished_participants_file = '03_finished_participants.csv'
finished_participants = pd.read_csv(os.path.join('output', 'intermediate', finished_participants_file), sep='\t')
display(finished_participants.head())

,Unnamed: 0.1,Unnamed: 0,id,event_uuid,name,time,event,data,participant_id,participant_token,participant_state,data_parsed,document_id
0,0,164,276174.0,cb251ecc-b22b-4ef9-af15-415f3ab90c25,Prototype,2025-03-07 21:32:17.982000,USER_AGENT,"""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN
1,1,166,276176.0,17b21821-b077-4b06-9db8-b341d36d0301,Prototype,2025-03-07 21:33:27.678000,USER_LOGIN,"{""username"": ""wolfova"", ""participant_token"": ""...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'username': 'wolfova', 'participant_token': '...",NaN
2,2,171,276181.0,c55551a8-7950-417a-aa64-9c323aaee006,Prototype,2025-03-07 21:33:49.699000,DOCUMENT_JOIN,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'user_id': '67cb6627285fac3f07bcf59b', 'usern...",67c97527285fac3f07bcf587
3,3,172,276182.0,65ca58d2-568f-4b16-af2d-718c2e633485,Prototype,2025-03-07 21:33:49.808000,DOCUMENT_VIEW,"{""user_id"": ""67cb6627285fac3f07bcf59b"", ""usern...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'user_id': '67cb6627285fac3f07bcf59b', 'usern...",67c97527285fac3f07bcf587
4,4,173,276183.0,20b1b819-a53b-4619-87e9-fe11dcba2855,Prototype,2025-03-07 21:33:59.815000,DOCUMENT_TEXT_UPDATE,"{""text"": ""<p>Write your text here</p>"", ""user_...",1376,a5bbafd1-4ef0-49e4-95bd-65170953633c,finished,"{'text': '<p>Write your text here</p>', 'user_...",67c97527285fac3f07bcf587


In [39]:
# method to read mongo database collection
def read_collection(file_path):

    # Step 1: Try reading as a JSON array
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except json.JSONDecodeError:
        # Step 2: If it's newline-delimited JSON
        with open(file_path, 'r', encoding='utf-8') as f:
            data = [json.loads(line) for line in f]

    # Step 3: Clean MongoDB special fields ($oid, $date, etc.)
    def extract_mongo_specials(obj):
        if isinstance(obj, dict):
            if "$oid" in obj:
                return obj["$oid"]
            if "$date" in obj:
                return obj["$date"]
            return {k: extract_mongo_specials(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [extract_mongo_specials(i) for i in obj]
        return obj

    cleaned_data = [extract_mongo_specials(doc) for doc in data]

    # Step 4: Flatten into DataFrame
    df = pd.json_normalize(cleaned_data)

    return df

In [40]:
# read agents collection
file_path = os.path.join('input/database', 'agent.json')
agents_df = read_collection(file_path)

# read tasks collection
task_path = os.path.join('input/database', 'task.json')
task_df = read_collection(task_path)

# read document collection
document_path = os.path.join('input/database', 'document.json')
documents_df = read_collection(document_path)

In [41]:
# parse generic event

def parse_json_field(event_df, json_field):
    #transform data from str values to dict values
    event_df[json_field + "_parsed"] = event_df[json_field].apply(json.loads)
    #build dataframe from dict values
    event_json_df = pd.json_normalize(event_df[json_field + "_parsed"])
    #reset indexes to concat the dataframes along axis 1 (1:1)
    event_df.reset_index(drop=True, inplace=True)
    event_json_df.reset_index(drop=True, inplace=True)
    return event_json_df

def parse_generic_event(generic_interactions, event = None):
    if event:
        if isinstance(event, list):
            event_df = generic_interactions[generic_interactions['event'].isin(event)]
        else:
            event_df = generic_interactions[generic_interactions['event'] == event]
    else:
        event_df = generic_interactions.copy()
    event_df = event_df[['id', 'time', 'participant_id', 'data', 'event', 'event_uuid', 'participant_token']]
    event_df['time'] = event_df['time'].astype(str)
    event_json_df = parse_json_field(event_df, 'data')
    return pd.concat([event_df, event_json_df], axis=1)

def parse_keyboard_event(keyboard_interactions):
    event_df = keyboard_interactions.copy()
    event_df['time'] = event_df['time'].astype(str)
    event_json_df = parse_json_field(event_df, 'meta_data')
    return pd.concat([event_df, event_json_df], axis=1)

In [42]:
# Removed interaction events that happened while interviewing (task has been completed already)
def remove_events_while_interviewing(df, remove_arr, time_field = "from_time"):
    format = '%Y-%m-%d %H:%M:%S.%f'
    dfn = df.copy()

    for t in remove_arr:
        participant_id = t[0]
        time = t[1]
        time = datetime.strptime(time, format)

        participant_mask = dfn['participant_id'] == participant_id
        time_mask = dfn[time_field] > time

        entries = dfn[participant_mask & time_mask]
        #display(entries.index)
        dfn = dfn.drop(entries.index)

    return dfn

In [43]:
# Convert to a list of rows
rows = []
for group_id, group in participants_groups.items():
    for i, user in enumerate(group["users"]):
        rows.append({
            "group_id": int(group_id),
            "user": user,
            "user_id": group["user_ids"][i],
            "document": group["documents"][0]
        })

# Create DataFrame our of participants_groups
groups_df = pd.DataFrame(rows)

display(groups_df)

,group_id,user,user_id,document
0,1,1379,67ce982f285fac3f07bcf5e7,67ce983b285fac3f07bcf5e8
1,1,1380,67ce9cb4285fac3f07bcf5ec,67ce983b285fac3f07bcf5e8
2,2,1376,67cb6627285fac3f07bcf59b,67c97527285fac3f07bcf587
3,2,1377,67cb6623285fac3f07bcf59a,67c97527285fac3f07bcf587
4,3,1378,67cc9f69285fac3f07bcf5c8,67c88f88285fac3f07bcf585
5,3,1374,67cdfa30285fac3f07bcf5cc,67c88f88285fac3f07bcf585
6,4,1383,67bc761a7a670b68746905a0,67d43b390b9539e69a3a86b9
7,4,1384,67d444dd0b9539e69a3a86bb,67d43b390b9539e69a3a86b9
8,4,1385,67bc43607a670b6874690577,67d43b390b9539e69a3a86b9
9,5,1387,67d5d8120b9539e69a3a86d7,67d5d8160b9539e69a3a86d8


## Comments

In [44]:
# COMMENT POST AND ADD
# number of added tasks
display("Cards ADDED:", len(finished_participants[finished_participants["event"] == "CARD_ADD"]))
# created comment event
display("Comments ADDED:", len(finished_participants[finished_participants["event"] == "COMMENT_ADD"]))
display("Comments POSTED:", len(finished_participants[finished_participants["event"] == "COMMENT_POST"]))
display(finished_participants[finished_participants["event"] == "COMMENT_POST"]["data"].to_list())


'Cards ADDED:'

39

'Comments ADDED:'

102

'Comments POSTED:'

71

['{"card": {"id": "XQt5XrkOapXdkdaV-J3Cn", "data": {"date": 1741593867387, "text": "Which topic?", "user": {"id": "67ce9cb4285fac3f07bcf5ec", "tag": "@mrnexeon", "name": "mrnexeon", "type": "you", "online": true, "picture": "/collab-editor/user2.svg", "author_id": ""}}, "mode": "comment", "type": "comment", "state": "POSTED", "typing": [], "history": [{"time": 1741593867387, "state": "NEW"}, {"time": 1741593872519, "state": "POSTED"}], "replies": [], "suggestion": null}, "users": {}, "agents": {}, "card_id": "XQt5XrkOapXdkdaV-J3Cn", "user_id": "67ce9cb4285fac3f07bcf5ec", "username": "mrnexeon", "document_id": "67ce983b285fac3f07bcf5e8", "marked_text": "essay", "comment_text": "Which topic?", "participant_token": "ecc027d0-42cd-42a1-bc2e-990b5516565b"}',
 '{"card": {"id": "CKT81i4uQciQ54og0mN8o", "data": {"date": 1741593883737, "text": "Yup, let\'s do that. Wanna start?", "user": {"id": "67ce9cb4285fac3f07bcf5ec", "tag": "@mrnexeon", "name": "mrnexeon", "type": "you", "online": true, "p

In [45]:
# review replies
reply_df = finished_participants[finished_participants["event"] == "REPLY_POST"]["data"].apply(json.loads).pipe(pd.json_normalize)
mentioned_agent_df = reply_df[reply_df['reply_text'].apply(lambda x: not re.search("(?<=^|(?<=[^a-zA-Z0-9-_\.]))@([A-Za-z]+[A-Za-z0-9_]+)", x) == None)]
display(finished_participants[finished_participants["event"] == "REPLY_POST"])
display(reply_df["reply_text"].to_list())

display(mentioned_agent_df)
display(mentioned_agent_df['user_id'])
display(mentioned_agent_df['card.data.user.id'])
display(mentioned_agent_df['card.replies'])

print("Reply POSTED: ", len(reply_df))
print("Mentioned agent (@...): ", len(mentioned_agent_df))
print("Percentage mentioned: ", len(mentioned_agent_df) * 100 / len(reply_df))

#print("Reply posted per card aka. turns: ")
#print(reply_df["card_id"].value_counts().describe())

,Unnamed: 0.1,Unnamed: 0,id,event_uuid,name,time,event,data,participant_id,participant_token,participant_state,data_parsed,document_id
368,370,547,276557.0,b2460698-34d6-4c05-84b3-4bff42f8ac2a,Prototype,2025-03-09 21:18:20.550000,REPLY_POST,"{""card"": {""id"": ""XzJllbkbnRCR-EZRbYc37"", ""data...",1374,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,finished,"{'card': {'id': 'XzJllbkbnRCR-EZRbYc37', 'data...",67c88f88285fac3f07bcf585
382,384,561,276571.0,dd44004a-d7f9-47bc-a6f5-3cd07f1e27ea,Prototype,2025-03-09 21:19:46.103000,REPLY_POST,"{""card"": {""id"": ""XzJllbkbnRCR-EZRbYc37"", ""data...",1374,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,finished,"{'card': {'id': 'XzJllbkbnRCR-EZRbYc37', 'data...",67c88f88285fac3f07bcf585
390,392,569,276579.0,2dcc9796-81fa-4cbe-83c2-8af8ed02d4a2,Prototype,2025-03-09 21:20:36.914000,REPLY_POST,"{""card"": {""id"": ""XzJllbkbnRCR-EZRbYc37"", ""data...",1374,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,finished,"{'card': {'id': 'XzJllbkbnRCR-EZRbYc37', 'data...",67c88f88285fac3f07bcf585
573,575,752,276762.0,da8b1c5f-60c2-491f-ba15-63ab628d6871,Prototype,2025-03-10 08:04:54.116000,REPLY_POST,"{""card"": {""id"": ""XQt5XrkOapXdkdaV-J3Cn"", ""data...",1379,6573f86e-0207-4286-b036-a83f25a630c3,finished,"{'card': {'id': 'XQt5XrkOapXdkdaV-J3Cn', 'data...",67ce983b285fac3f07bcf5e8
583,585,762,276772.0,75b76276-1816-412c-9da9-611d7c0e1543,Prototype,2025-03-10 08:05:19.912000,REPLY_POST,"{""card"": {""id"": ""XQt5XrkOapXdkdaV-J3Cn"", ""data...",1380,ecc027d0-42cd-42a1-bc2e-990b5516565b,finished,"{'card': {'id': 'XQt5XrkOapXdkdaV-J3Cn', 'data...",67ce983b285fac3f07bcf5e8
595,597,774,276784.0,487fcc46-635f-4f97-adb2-d506bdaa6f4c,Prototype,2025-03-10 08:05:54.368000,REPLY_POST,"{""card"": {""id"": ""XQt5XrkOapXdkdaV-J3Cn"", ""data...",1379,6573f86e-0207-4286-b036-a83f25a630c3,finished,"{'card': {'id': 'XQt5XrkOapXdkdaV-J3Cn', 'data...",67ce983b285fac3f07bcf5e8
599,601,778,276788.0,fc642c10-7a9f-43af-b7f4-4d9bef62bb1e,Prototype,2025-03-10 08:06:06.791000,REPLY_POST,"{""card"": {""id"": ""XQt5XrkOapXdkdaV-J3Cn"", ""data...",1380,ecc027d0-42cd-42a1-bc2e-990b5516565b,finished,"{'card': {'id': 'XQt5XrkOapXdkdaV-J3Cn', 'data...",67ce983b285fac3f07bcf5e8
637,639,816,276826.0,7f4b8af1-17b9-42d9-b05b-ad2e16bbd8f5,Prototype,2025-03-10 08:07:08.130000,REPLY_POST,"{""card"": {""id"": ""XQt5XrkOapXdkdaV-J3Cn"", ""data...",1379,6573f86e-0207-4286-b036-a83f25a630c3,finished,"{'card': {'id': 'XQt5XrkOapXdkdaV-J3Cn', 'data...",67ce983b285fac3f07bcf5e8
734,736,913,276923.0,055e4dc7-7fd5-4242-a5ec-099574beb550,Prototype,2025-03-10 08:12:27.417000,REPLY_POST,"{""card"": {""id"": ""ibJj8EeObiQnh6gJucY0X"", ""data...",1380,ecc027d0-42cd-42a1-bc2e-990b5516565b,finished,"{'card': {'id': 'ibJj8EeObiQnh6gJucY0X', 'data...",67ce983b285fac3f07bcf5e8
788,790,967,276977.0,d3358c11-9596-4c86-b328-49d1eadf7463,Prototype,2025-03-10 08:13:57.183000,REPLY_POST,"{""card"": {""id"": ""h10aZVeG_1lYcZ-OvWQUE"", ""data...",1380,ecc027d0-42cd-42a1-bc2e-990b5516565b,finished,"{'card': {'id': 'h10aZVeG_1lYcZ-OvWQUE', 'data...",67ce983b285fac3f07bcf5e8


['<span class="tag-mention">@aiAI Max</span> <br>Could you please make your answer shorter so that I can put it in the text.',
 '<div><span class="tag-mention">@aiAI Max</span> , yes I do. Give me some papers and build them in the essay</div>',
 '<div><span class="tag-mention">@aiAI Max</span>, ok, make it so that I can put it in the text</div>',
 '<span class="tag-mention">@aiauthor</span>&nbsp;give us topics ideas for an essay',
 'I like Health and Wellness in the Digital Age',
 "ok, let's stick with this one<div><br></div>",
 '<span style="background-color: rgb(221, 221, 221);">Ok : )</span><div><span style="background-color: rgb(221, 221, 221);"><br></span></div>',
 '<span class="tag-mention">@aiauthor</span>&nbsp;give us the structure of the essay "<span style="font-weight: bolder; font-family: &quot;Helvetica Neue&quot;, Helvetica, Arial, sans-serif; text-align: center; white-space-collapse: preserve;">Health and Wellness in the Digital Age"</span>',
 'How about If I take the Bod

,card_id,user_id,reply_id,username,reply_text,document_id,marked_text,participant_token,card.id,card.data.date,...,agents.6888c9af87cf5fec60dc8081.picture,agents.6888c9af87cf5fec60dc8081.author_id,agents.688b7f6b87cf5fec60dc80b7.id,agents.688b7f6b87cf5fec60dc80b7.tag,agents.688b7f6b87cf5fec60dc80b7.name,agents.688b7f6b87cf5fec60dc80b7.type,agents.688b7f6b87cf5fec60dc80b7.online,agents.688b7f6b87cf5fec60dc80b7.picture,agents.688b7f6b87cf5fec60dc80b7.author_id,card.overlapTasks.688b8b8b87cf5fec60dc80c0
0,XzJllbkbnRCR-EZRbYc37,67cdfa30285fac3f07bcf5cc,xwctzrwyYJfjdjZ6gAXXF,Ivan Khrop,"<span class=""tag-mention"">@aiAI Max</span> <br...",67c88f88285fac3f07bcf585,The rapid development of Artificial Intelligen...,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,XzJllbkbnRCR-EZRbYc37,1.741555e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,XzJllbkbnRCR-EZRbYc37,67cdfa30285fac3f07bcf5cc,OvYXJkTV1ae6MBFfl5hTL,Ivan Khrop,"<div><span class=""tag-mention"">@aiAI Max</span...",67c88f88285fac3f07bcf585,The rapid development of Artificial Intelligen...,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,XzJllbkbnRCR-EZRbYc37,1.741555e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,XzJllbkbnRCR-EZRbYc37,67cdfa30285fac3f07bcf5cc,5oBdwK5Q1QkRBHrgS9EGT,Ivan Khrop,"<div><span class=""tag-mention"">@aiAI Max</span...",67c88f88285fac3f07bcf585,The rapid development of Artificial Intelligen...,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,XzJllbkbnRCR-EZRbYc37,1.741555e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,XQt5XrkOapXdkdaV-J3Cn,67ce982f285fac3f07bcf5e7,ZUM74QuTz8jaRKDelAYNf,galina.akhunzhanova@uni-bayreuth.de,"<span class=""tag-mention"">@aiauthor</span>&nbs...",67ce983b285fac3f07bcf5e8,essay,6573f86e-0207-4286-b036-a83f25a630c3,XQt5XrkOapXdkdaV-J3Cn,1.741594e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,XQt5XrkOapXdkdaV-J3Cn,67ce982f285fac3f07bcf5e7,0BZC6IseVK0ItTsZJwyHn,galina.akhunzhanova@uni-bayreuth.de,"<span class=""tag-mention"">@aiauthor</span>&nbs...",67ce983b285fac3f07bcf5e8,essay,6573f86e-0207-4286-b036-a83f25a630c3,XQt5XrkOapXdkdaV-J3Cn,1.741594e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,18-cROhkXZwVmH9qkyXtx,67ce982f285fac3f07bcf5e7,mDSCleY4I76ZB9liBPLEC,galina.akhunzhanova@uni-bayreuth.de,"<span class=""tag-mention"">@aiauthor</span>&nbs...",67ce983b285fac3f07bcf5e8,"In conclusion, the integration of digital tech...",6573f86e-0207-4286-b036-a83f25a630c3,18-cROhkXZwVmH9qkyXtx,1.741596e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,18-cROhkXZwVmH9qkyXtx,67ce982f285fac3f07bcf5e7,Y92gXaI8XGI3KzVAYix7w,galina.akhunzhanova@uni-bayreuth.de,"<span class=""tag-mention"">@aiauthor</span>&nbs...",67ce983b285fac3f07bcf5e8,"In conclusion, the integration of digital tech...",6573f86e-0207-4286-b036-a83f25a630c3,18-cROhkXZwVmH9qkyXtx,1.741596e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,2TPbqrhSqTafGZwjVQFi4,67ce9cb4285fac3f07bcf5ec,M5rJ1etJc-dI5eWszjxfb,mrnexeon,"I added <span class=""tag-mention"">@aiBob</span...",67ce983b285fac3f07bcf5e8,Body,ecc027d0-42cd-42a1-bc2e-990b5516565b,2TPbqrhSqTafGZwjVQFi4,1.741596e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,JdQKwfkzfixwA-bBM8KrJ,67cdfa30285fac3f07bcf5cc,nn7N8GMpIA_rQm5FpIsSh,Ivan Khrop,"<div><span class=""tag-mention"">@aiauthor</span...",67c88f88285fac3f07bcf585,From streamlining administrative tasks to enha...,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,JdQKwfkzfixwA-bBM8KrJ,1.741613e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,tVDXKA4_o6cP_9KXetG4G,67d5d8120b9539e69a3a86d7,FuLSkdGboodkCHolCnJM3,Ilya School,"<span class=""tag-mention"">@aiGeorge</span> Hm,...",67d5d8160b9539e69a3a86d8,"the other extreme, completely unstructured in...",5b6b66f7-99db-48a1-b61a-0eaf97a1c8fc,tVDXKA4_o6cP_9KXetG4G,1.742068e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


0     67cdfa30285fac3f07bcf5cc
1     67cdfa30285fac3f07bcf5cc
2     67cdfa30285fac3f07bcf5cc
3     67ce982f285fac3f07bcf5e7
7     67ce982f285fac3f07bcf5e7
16    67ce982f285fac3f07bcf5e7
17    67ce982f285fac3f07bcf5e7
18    67ce9cb4285fac3f07bcf5ec
21    67cdfa30285fac3f07bcf5cc
23    67d5d8120b9539e69a3a86d7
24    67d5d8120b9539e69a3a86d7
25    67d5d8120b9539e69a3a86d7
26    6880e0f187cf5fec60dc7ffe
27    6880e0f187cf5fec60dc7ffe
30    68835b0587cf5fec60dc8020
32    68835b0587cf5fec60dc8020
34    6888bbce87cf5fec60dc8072
35    6888bbce87cf5fec60dc8072
36    6888bbce87cf5fec60dc8072
43    688b1c7887cf5fec60dc80b0
46    688bdb6387cf5fec60dc80d0
48    6888eec887cf5fec60dc8085
49    6888eec887cf5fec60dc8085
Name: user_id, dtype: object

0     67cdfa30285fac3f07bcf5cc
1     67cdfa30285fac3f07bcf5cc
2     67cdfa30285fac3f07bcf5cc
3     67ce9cb4285fac3f07bcf5ec
7     67ce9cb4285fac3f07bcf5ec
16    67ce982f285fac3f07bcf5e7
17    67ce982f285fac3f07bcf5e7
18    67ce9cb4285fac3f07bcf5ec
21    67cdfa30285fac3f07bcf5cc
23    67d5d88f0b9539e69a3a86dc
24    67d5d88f0b9539e69a3a86dc
25    67d5d88f0b9539e69a3a86dc
26    6880e0f187cf5fec60dc7ffe
27    6880e0f187cf5fec60dc7ffe
30    68835b0587cf5fec60dc8020
32    68835b0587cf5fec60dc8020
34    6888bbce87cf5fec60dc8072
35    6888bbce87cf5fec60dc8072
36    6888bbce87cf5fec60dc8072
43    688b1c7887cf5fec60dc80b0
46    688bdb6387cf5fec60dc80d0
48                   ai-author
49    6888eec887cf5fec60dc8085
Name: card.data.user.id, dtype: object

0     [{'id': 'mKXUvYDt64Vu7lTGJaNFT', 'data': {'ai'...
1     [{'id': 'mKXUvYDt64Vu7lTGJaNFT', 'data': {'ai'...
2     [{'id': 'mKXUvYDt64Vu7lTGJaNFT', 'data': {'ai'...
3                                                    []
7     [{'id': 'ZUM74QuTz8jaRKDelAYNf', 'data': {'tex...
16                                                   []
17    [{'id': 'mDSCleY4I76ZB9liBPLEC', 'data': {'tex...
18                                                   []
21    [{'id': 'oe-9mbbRS3gaN18k7FXV0', 'data': {'ai'...
23    [{'id': 'Spr7U8KkbYEzyGvMQCkhe', 'data': {'ai'...
24    [{'id': 'Spr7U8KkbYEzyGvMQCkhe', 'data': {'ai'...
25    [{'id': 'Spr7U8KkbYEzyGvMQCkhe', 'data': {'ai'...
26                                                   []
27    [{'id': '59DZupqmlDTal5SFOWhyg', 'data': {'tex...
30    [{'id': 'c2BEu3whdI7D4xNttDjZB', 'data': {'ai'...
32    [{'id': '9K0htmFNosyL9Ze2T9Tpd', 'data': {'ai'...
34    [{'id': 'VHdtiFxJL_0sm6RzLL_QY', 'data': {'ai'...
35    [{'id': 'VHdtiFxJL_0sm6RzLL_QY', 'data': {

Reply POSTED:  50
Mentioned agent (@...):  23
Percentage mentioned:  46.0


- approving a comment is equal to closing a comment but with a positive meaning
- deleting comment is deleting comment
- accepting: append, replace, copy2clipboard

[R] Number of cases with accept / close

In [46]:
# number of approved cards
approved_cards = parse_generic_event(finished_participants, "CARD_APPROVE")

display(approved_cards.columns)
display("Approved cards:", approved_cards["id"].count())

approved_comments = approved_cards[approved_cards["type"] == "comment"]

approved_comments_by_creator = approved_comments[approved_comments["user_id"] == approved_comments["card.data.user.id"]]
approved_comments_by_others = approved_comments[approved_comments["user_id"] != approved_comments["card.data.user.id"]]
display("Approved comments:", approved_comments["id"].count())
display("Approved by creator:", approved_comments_by_creator["id"].count())
display("Approved by others:", approved_comments_by_others["id"].count())

Index(['id', 'time', 'participant_id', 'data', 'event', 'event_uuid',
       'participant_token', 'data_parsed', 'type', 'card_id', 'user_id',
       'username', 'document_id', 'participant_token', 'card.ai.prediction',
       'card.id', 'card.data.date', 'card.data.user.id', 'card.data.user.tag',
       'card.data.user.name', 'card.data.user.type', 'card.data.user.online',
       'card.data.user.picture', 'card.data.user.author_id', 'card.mode',
       'card.type', 'card.state', 'card.history', 'card.isLoading',
       'card.suggestion.id', 'card.suggestion.user',
       'card.overlapTasks.67cb68ff285fac3f07bcf5a8',
       'card.overlapTasks.67cb691d285fac3f07bcf5ac', 'card.ai.function'],
      dtype='object')

'Approved cards:'

7

'Approved comments:'

0

'Approved by creator:'

0

'Approved by others:'

0

In [47]:
def transform_reaction_times(df):
    df = transform_time(df, 'time')
    df["card.data.date"] = df["card.data.date"].apply(lambda x: datetime.utcfromtimestamp(x / 1000))
    return df

def compute_time_diff(df):
    diff = (df["time"] - df["card.data.date"])
    return diff.apply(lambda x: x.total_seconds())

In [48]:
# number of deleted comments
deleted_comments = parse_generic_event(finished_participants, "COMMENT_DELETE")
deleted_comments = transform_reaction_times(deleted_comments)

display(deleted_comments.columns)
display(deleted_comments)

deleted_comments_by_creator = deleted_comments[deleted_comments["user_id"] == deleted_comments["card.data.user.id"]]
deleted_comments_by_others = deleted_comments[deleted_comments["user_id"] != deleted_comments["card.data.user.id"]]

deleted_comments_human = deleted_comments[deleted_comments["card.data.user.id"] != "ai-author"]
deleted_comments_ai = deleted_comments[deleted_comments["card.data.user.id"] == "ai-author"]

deleted_comments_human_creator = deleted_comments_human[deleted_comments_human["user_id"] == deleted_comments_human["card.data.user.id"]]
deleted_comments_human_others = deleted_comments_human[deleted_comments_human["user_id"] != deleted_comments_human["card.data.user.id"]]


display("Time to reaction (COMMENT_DELETE) by CREATOR:")
diff_creator = compute_time_diff(deleted_comments_by_creator)
display(diff_creator.describe())

display("Time to reaction (COMMENT_DELETE) by OTHERS:")
diff_others = compute_time_diff(deleted_comments_by_others)
display(diff_others.describe())

display("Time to reaction (COMMENT_DELETE) COMBINED:")
diff = pd.concat([diff_creator, diff_others])
display(diff.describe())

display("Time to reaction (COMMENT_DELETE) created by HUMAN:")
diff_human = compute_time_diff(deleted_comments_human)
display(diff_human.describe())

display("Time to reaction (COMMENT_DELETE) created by AI:")
diff_ai = compute_time_diff(deleted_comments_ai)
display(diff_ai.describe())

display("Time to reaction (COMMENT_DELETE) created by Human, SELF reacted:")
diff_human_self = compute_time_diff(deleted_comments_human_creator)
display(diff_human_self.describe())

display("Time to reaction (COMMENT_DELETE) created by Human, OTHERS reacted:")
diff_human_others = compute_time_diff(deleted_comments_human_others)
display(diff_human_others.describe())

display("Deleted comments:", deleted_comments["id"].count())
display("Deleted by creator:", deleted_comments_by_creator["id"].count())
display(deleted_comments_by_creator)
display("Deleted by others:", deleted_comments_by_others["id"].count())
display("Deleted by others, creator was human: ", deleted_comments_by_others[deleted_comments_by_others["card.data.user.id"] != "ai-author"]["id"].count())
display("Deleted by others, creator was AI: ", deleted_comments_by_others[deleted_comments_by_others["card.data.user.id"] == "ai-author"]["id"].count())

Index(['id', 'time', 'participant_id', 'data', 'event', 'event_uuid',
       'participant_token', 'data_parsed', 'card_id', 'user_id', 'username',
       'document_id', 'participant_token', 'card.id', 'card.data.date',
       'card.data.text', 'card.data.user.id', 'card.data.user.tag',
       'card.data.user.name', 'card.data.user.type', 'card.data.user.online',
       'card.data.user.picture', 'card.data.user.author_id', 'card.mode',
       'card.type', 'card.state', 'card.typing', 'card.history',
       'card.replies', 'card.suggestion', 'card.source', 'card.taskId',
       'card.taskTitle', 'card.suggestion.id', 'card.suggestion.user',
       'card.likeStatus'],
      dtype='object')

,id,time,participant_id,data,event,event_uuid,participant_token,data_parsed,card_id,user_id,...,card.typing,card.history,card.replies,card.suggestion,card.source,card.taskId,card.taskTitle,card.suggestion.id,card.suggestion.user,card.likeStatus
0,277050.0,2025-03-10 08:15:46.921,1379,"{""card"": {""id"": ""h10aZVeG_1lYcZ-OvWQUE"", ""data...",COMMENT_DELETE,e8cf6c0e-df86-4ed6-abb5-29e24610bac8,6573f86e-0207-4286-b036-a83f25a630c3,"{'card': {'id': 'h10aZVeG_1lYcZ-OvWQUE', 'data...",h10aZVeG_1lYcZ-OvWQUE,67ce982f285fac3f07bcf5e7,...,[],"[{'time': 1741594381316, 'state': 'NEW'}, {'ti...","[{'id': '75MWuFnjcfxrKbUiPCve_', 'data': {'tex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,277319.0,2025-03-10 08:32:10.942,1379,"{""card"": {""id"": ""HDRrJll97j9BSVFDgbXsM"", ""data...",COMMENT_DELETE,5e77526e-77c7-49be-9090-13f50e18b534,6573f86e-0207-4286-b036-a83f25a630c3,"{'card': {'id': 'HDRrJll97j9BSVFDgbXsM', 'data...",HDRrJll97j9BSVFDgbXsM,67ce982f285fac3f07bcf5e7,...,[],"[{'time': 1741595322937.4114, 'state': 'POSTED...","[{'id': 'g7FotqM4Tkdif6Dg4zQTl', 'data': {'ai'...",NaN,remote,67cea2a4285fac3f07bcf5f8,Conclusion Writing Task,xopusrY3YDxzfvm2_g-ue,AI author,NaN
2,277323.0,2025-03-10 08:32:15.658,1379,"{""card"": {""id"": ""bWe12yg7mfhYgU4X00BAh"", ""data...",COMMENT_DELETE,f204e0c8-92b9-46c3-bf5e-4ff0ea6b9b8a,6573f86e-0207-4286-b036-a83f25a630c3,"{'card': {'id': 'bWe12yg7mfhYgU4X00BAh', 'data...",bWe12yg7mfhYgU4X00BAh,67ce982f285fac3f07bcf5e7,...,[],"[{'time': 1741595322937.4114, 'state': 'POSTED'}]","[{'id': 't8HpZmEAlf70lAYyn8TGq', 'data': {'ai'...",NaN,remote,67cea2a4285fac3f07bcf5f8,Conclusion Writing Task,NaN,NaN,NaN
3,277325.0,2025-03-10 08:32:19.197,1379,"{""card"": {""id"": ""963qXwEeVR11GoKh8Yhyy"", ""data...",COMMENT_DELETE,55784d6b-d878-4273-8010-dec7920ab545,6573f86e-0207-4286-b036-a83f25a630c3,"{'card': {'id': '963qXwEeVR11GoKh8Yhyy', 'data...",963qXwEeVR11GoKh8Yhyy,67ce982f285fac3f07bcf5e7,...,[],"[{'time': 1741595322937.4114, 'state': 'POSTED'}]","[{'id': 'VFToAf18sCTEzNQJyplws', 'data': {'ai'...",NaN,remote,67cea2a4285fac3f07bcf5f8,Conclusion Writing Task,NaN,NaN,NaN
4,278942.0,2025-03-17 15:07:23.024,1388,"{""card"": {""id"": ""Ccm6J_B5JSl4kq1WkrokH"", ""data...",COMMENT_DELETE,17ce93b1-5d6e-44b0-87e1-0edb191d4314,3b49f849-3081-4478-81ce-d65450662c47,"{'card': {'id': 'Ccm6J_B5JSl4kq1WkrokH', 'data...",Ccm6J_B5JSl4kq1WkrokH,67d82dc4edfe680c1b0f1642,...,[],"[{'time': 1742223469692.4097, 'state': 'POSTED'}]","[{'id': 'KWp3YCYjppaKhwJ3POCqx', 'data': {'ai'...",NaN,remote,67d83857edfe680c1b0f1667,Introduction and Conclusion Addition,NaN,NaN,NaN
5,280559.0,2025-07-28 18:02:51.233,2460,"{""card"": {""id"": ""FGaa84DVAZLj9oCPzaPwF"", ""data...",COMMENT_DELETE,00d9d99e-a1a3-4167-94f4-5fe3046dd6c9,4f4b17e2-dcb0-4391-a3de-8f3a8c3fac50,"{'card': {'id': 'FGaa84DVAZLj9oCPzaPwF', 'data...",FGaa84DVAZLj9oCPzaPwF,6887b97087cf5fec60dc8060,...,"[{'id': '6887772287cf5fec60dc8036', 'tag': '@a...","[{'time': 1753725734373, 'state': 'NEW'}, {'ti...",[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,280752.0,2025-07-29 11:55:48.585,2460,"{""card"": {""id"": ""reHdO1wu97rfiuqOKxs_W"", ""data...",COMMENT_DELETE,2d7af653-d016-4c95-8594-2c613ff23e77,4f4b17e2-dcb0-4391-a3de-8f3a8c3fac50,"{'card': {'id': 'reHdO1wu97rfiuqOKxs_W', 'data...",reHdO1wu97rfiuqOKxs_W,6887b97087cf5fec60dc8060,...,[],"[{'time': 1753790096019, 'state': 'NEW'}, {'ti...","[{'id': '1Jx7xhQRPdsmUUGGbi2KG', 'data': {'ai'...",NaN,NaN,NaN,NaN,sgSd3A9PGxzzv-gw3MrDq,Fun facts,NaN
7,280795.0,2025-07-29 11:59:50.532,2460,"{""card"": {""id"": ""BSg_adC1_t7w9Ut31cwdh"", ""data...",COMMENT_DELETE,186a3dc8-d4c4-4b3c-ad4c-1297e5b0f7b5,4f4b17e2-dcb0-4391-a3de-8f3a8c3fac50,"{'card': {'id': 'BSg_adC1_t7w9Ut31cwdh', 'data...",BSg_adC1_t7w9Ut31cwdh,6887b97087cf5fec60dc8060,...,[],"[{'time': 1753725709467.5225, 'state': 'POSTED'}]","[{'id': 'fE_uen42QO9ZmjC_Osc8i', 'data': {'ai'...",NaN,remote,6887bac987cf5fec60dc8062,Enhance Literary Focus,NaN,NaN,NaN
8,280846.0,2025-07-29 12:04:41.835,2460,"{""card"": {""id"": ""sYPT

'Time to reaction (COMMENT_DELETE) by CREATOR:'

count      5.000000
mean     106.410000
std       99.442066
min       23.937000
25%       36.860000
50%       52.566000
75%      165.605000
max      253.082000
dtype: float64

'Time to reaction (COMMENT_DELETE) by OTHERS:'

...

10

'Deleted by others, creator was human: '

5

'Deleted by others, creator was AI: '

5

In [49]:
# number of appended suggestions
appended_suggestions = parse_generic_event(finished_participants, "SUGGESTION_INSERT_AFTER")
appended_suggestions = transform_reaction_times(appended_suggestions)

display(appended_suggestions.columns)
display(appended_suggestions)

appended_suggestion_by_creator = appended_suggestions[appended_suggestions["user_id"] == appended_suggestions["card.data.user.id"]]
appended_suggestion_by_others = appended_suggestions[appended_suggestions["user_id"] != appended_suggestions["card.data.user.id"]]

appended_suggestion_human = appended_suggestions[appended_suggestions["card.data.user.id"] != "ai-author"]
appended_suggestion_ai = appended_suggestions[appended_suggestions["card.data.user.id"] == "ai-author"]

appended_suggestion_ai_remote = appended_suggestion_ai[appended_suggestion_ai["card.source"] == "remote"]
appended_suggestion_ai_mention = appended_suggestion_ai[appended_suggestion_ai["card.source"] != "remote"]

appended_suggestion_human_creator = appended_suggestion_human[appended_suggestion_human["user_id"] == appended_suggestion_human["card.data.user.id"]]
appended_suggestion_human_others = appended_suggestion_human[appended_suggestion_human["user_id"] != appended_suggestion_human["card.data.user.id"]]

display("Time to reaction (APPEND) by CREATOR:")
diff_creator = compute_time_diff(appended_suggestion_by_creator)
display(diff_creator.describe())

display("Time to reaction (APPEND) by OTHERS:")
diff_others = compute_time_diff(appended_suggestion_by_others)
display(diff_others.describe())

display("Time to reaction (APPEND) COMBINED:")
diff = pd.concat([diff_creator, diff_others])
display(diff.describe())

display("Time to reaction (APPEND) created by HUMAN:")
diff_human = compute_time_diff(appended_suggestion_human)
display(diff_human.describe())

display("Time to reaction (APPEND) created by AI:")
diff_ai = compute_time_diff(appended_suggestion_ai)
display(diff_ai.describe())

display("Time to reaction (APPEND) created by Human, SELF reacted:")
diff_human_self = compute_time_diff(appended_suggestion_human_creator)
display(diff_human_self.describe())

display("Time to reaction (APPEND) created by Human, OTHERS reacted:")
diff_human_others = compute_time_diff(appended_suggestion_human_others)
display(diff_human_others.describe())

#print(appended_suggestion_ai[["card.data.date", "time"]])

display("Appended suggestions:", appended_suggestions["id"].count())
display("Appended by creator:", appended_suggestion_by_creator["id"].count())
display(appended_suggestion_by_creator[["user_id", "card.data.user.id"]])
display("Appended by others, total:", appended_suggestion_by_others["id"].count())
display("Appended by others, creator was human: ", appended_suggestion_by_others[appended_suggestion_by_others["card.data.user.id"] != "ai-author"]["id"].count())
display("Appended by others, creator was AI: ", appended_suggestion_by_others[appended_suggestion_by_others["card.data.user.id"] == "ai-author"]["id"].count())

display(appended_suggestion_by_others[["user_id", "card.data.user.id"]])

Index(['id', 'time', 'participant_id', 'data', 'event', 'event_uuid',
       'participant_token', 'data_parsed', 'text', 'card_id', 'user_id',
       'username', 'document_id', 'marked_text', 'suggestion_id',
       'refined_by_user', 'participant_token', 'card.id', 'card.data.date',
       'card.data.text', 'card.data.user.id', 'card.data.user.tag',
       'card.data.user.name', 'card.data.user.type', 'card.data.user.online',
       'card.data.user.picture', 'card.mode', 'card.type', 'card.state',
       'card.source', 'card.taskId', 'card.typing', 'card.history',
       'card.replies', 'card.taskTitle', 'reply.id',
       'reply.data.ai.data.prediction', 'reply.data.text', 'reply.data.time',
       'reply.data.user.id', 'reply.data.user.tag', 'reply.data.user.name',
       'reply.data.user.type', 'reply.data.user.online',
       'reply.data.user.picture', 'reply.state', 'reply.source',
       'reply.history', 'reply.replies', 'card_type', 'card.ai.prediction',
       'card.data.user.

,id,time,participant_id,data,event,event_uuid,participant_token,data_parsed,text,card_id,...,card.ai.prediction,card.data.user.author_id,card.isLoading,card.suggestion,reply.data.user.author_id,card.likeStatus,card.overlapTasks.67d83435edfe680c1b0f1656,card.suggestion.id,card.suggestion.user,card.ai.function
0,276239.0,2025-03-07 21:42:34.724,1377,"{""card"": {""id"": ""h9dJoEjrePKz1x9XOxrxx"", ""data...",SUGGESTION_INSERT_AFTER,31658bce-0fbf-4559-9541-0242b1f07006,2aac5afe-a3af-4135-a1a3-78adf87ae9d8,"{'card': {'id': 'h9dJoEjrePKz1x9XOxrxx', 'data...","In this essay, we will delve into the various ...",h9dJoEjrePKz1x9XOxrxx,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,276285.0,2025-03-07 21:44:21.621,1376,"{""card"": {""id"": ""5Plaf8nQ31ghtWn4OAFbg"", ""data...",SUGGESTION_INSERT_AFTER,b0449168-c323-4422-87a9-b21ded2e7520,a5bbafd1-4ef0-49e4-95bd-65170953633c,"{'card': {'id': '5Plaf8nQ31ghtWn4OAFbg', 'data...",The aim of this part of the document is to thr...,5Plaf8nQ31ghtWn4OAFbg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,276320.0,2025-03-07 21:45:43.489,1377,"{""card"": {""ai"": {""prediction"": "" \n\nFirst and...",SUGGESTION_INSERT_AFTER,8c4b3c43-7323-46e2-98ea-745c70df289d,2aac5afe-a3af-4135-a1a3-78adf87ae9d8,{'card': {'ai': {'prediction': ' First and f...,"\n\nFirst and foremost, AI has the ability to...",8A2-OJj7uKw6ckP7d2DVP,...,"\n\nFirst and foremost, AI has the ability to...",,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,276347.0,2025-03-07 21:46:31.484,1376,"{""card"": {""id"": ""T1140AO9etlFPy5sX0me-"", ""data...",SUGGESTION_INSERT_AFTER,ce9eb039-b494-481b-8dbf-5329df65e520,a5bbafd1-4ef0-49e4-95bd-65170953633c,"{'card': {'id': 'T1140AO9etlFPy5sX0me-', 'data...",The objective of the document is to provide a ...,T1140AO9etlFPy5sX0me-,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,276387.0,2025-03-07 21:49:31.354,1376,"{""card"": {""id"": ""EiPiEDL-QXWG5qiQOj9tm"", ""data...",SUGGESTION_INSERT_AFTER,07f107d3-e7ba-4491-9556-25bd48034de4,a5bbafd1-4ef0-49e4-95bd-65170953633c,"{'card': {'id': 'EiPiEDL-QXWG5qiQOj9tm', 'data...",Document Goal: \n\nThe primary objective of th...,EiPiEDL-QXWG5qiQOj9tm,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,283056.0,2025-08-14 18:16:38.280,2471,"{""card"": {""id"": ""VHZhwRvxTK_pt97hTMcLg"", ""data...",SUGGESTION_INSERT_AFTER,92ee4a12-13f1-438e-9d2d-81c8916252d4,41b03f59-088a-4ca7-b174-44980d1d8d78,"{'card': {'id': 'VHZhwRvxTK_pt97hTMcLg', 'data...",The revised text you've provided is quite clea...,VHZhwRvxTK_pt97hTMcLg,...,NaN,,NaN,NaN,689e22d5e90ffc190a96c2e0,NaN,NaN,NaN,NaN,NaN
68,284402.0,2025-08-22 17:14:32.718,2475,"{""card"": {""id"": ""oucHjS_N0NE6Ec1wd4bI-"", ""data...",SUGGESTION_INSERT_AFTER,6f15c591-b7ac-40cb-b47b-3d5a77772f9c,5d9124de-67e0-4bd1-80e6-ec33aa775d1a,"{'card': {'id': 'oucHjS_N0NE6Ec1wd4bI-', 'data...",To create a comprehensive lesson plan for the ...,oucHjS_N0NE6Ec1wd4bI-,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69,284430.0,2025-08-22 17:19:55.096,2475,"{""card"": {""id"": ""0_vn2aveJwiAmNBnn1HPB"", ""data...",SUGGESTION_INSERT_AFTER,c9d50ccb-5813-4f22-9af2-6e72fc3450f8,5d9124de-67e0-4bd1-80e6-ec33aa775d1a,"{'card': {'id': '0_vn2aveJwiAmNBnn1HPB', 'data...",Here is a detailed lesson plan for a graduate ...,0_vn2aveJwiAmNBnn1HPB,...,NaN,,NaN,NaN,6889e71f87cf5fec60dc8094,NaN,NaN,NaN,NaN,NaN
70,284470.0,2025-08-22 17:36:11.852,2475,"{""card"": {""id"": ""T6dovEXM5kHzPLUczuYjE"", ""data...",SUGGESTION_INSERT_AFTER,9c465731-1d58-4881-8f6f-6169ff1f22fa,5d9124de-67e0-4bd1-80e6-ec33aa775d1a,"{'card': {'id': 'T6dovEXM5kHzPLUczuYjE', 'data...",The literature references provided focus on re...,T6dovEXM5kHzPLUczuYjE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


'Time to reaction (APPEND) by CREATOR:'

count     46.000000
mean      58.396087
std       87.395480
min        4.133000
25%       14.981250
50%       28.103500
75%       67.599500
max      487.711000
dtype: float64

'Time to reaction (APPEND) by OTHERS:'

...

'Appended by others, creator was human: '

19

'Appended by others, creator was AI: '

7

,user_id,card.data.user.id
0,67cb6623285fac3f07bcf59a,ai-author
1,67cb6627285fac3f07bcf59b,ai-author
3,67cb6627285fac3f07bcf59b,ai-author
4,67cb6627285fac3f07bcf59b,ai-author
8,67ce982f285fac3f07bcf5e7,ai-author
10,67d5d8120b9539e69a3a86d7,67d5d88f0b9539e69a3a86dc
11,67d5d8120b9539e69a3a86d7,67d5d88f0b9539e69a3a86dc
12,67d82dc4edfe680c1b0f1642,67d82f6dedfe680c1b0f164a
13,67d82dc4edfe680c1b0f1642,67d82f6dedfe680c1b0f164a
14,67d82e0aedfe680c1b0f1643,67d82fbfedfe680c1b0f164e


In [50]:
# number of appended suggestions
replaced_suggestions = parse_generic_event(finished_participants, "SUGGESTION_TAKE_OVER")
replaced_suggestions = transform_reaction_times(replaced_suggestions)

display(replaced_suggestions.columns)
display(replaced_suggestions)

replaced_suggestion_by_creator = replaced_suggestions[replaced_suggestions["user_id"] == replaced_suggestions["card.data.user.id"]]
replaced_suggestion_by_others = replaced_suggestions[replaced_suggestions["user_id"] != replaced_suggestions["card.data.user.id"]]

replaced_suggestion_human = replaced_suggestions[replaced_suggestions["card.data.user.id"] != "ai-author"]
replaced_suggestion_ai = replaced_suggestions[replaced_suggestions["card.data.user.id"] == "ai-author"]

replaced_suggestion_human_creator = replaced_suggestion_human[replaced_suggestion_human["user_id"] == replaced_suggestion_human["card.data.user.id"]]
replaced_suggestion_human_others = replaced_suggestion_human[replaced_suggestion_human["user_id"] != replaced_suggestion_human["card.data.user.id"]]

display("Time to reaction (REPLACE) by CREATOR:")
diff_creator = compute_time_diff(replaced_suggestion_by_creator)
display(diff_creator.describe())

display("Time to reaction (REPLACE) by OTHERS:")
diff_others = compute_time_diff(replaced_suggestion_by_others)
display(diff_others.describe())

display("Time to reaction (REPLACE) COMBINED:")
diff = pd.concat([diff_creator, diff_others])
display(diff.describe())

display("Time to reaction (REPLACE) created by HUMAN:")
diff_human = compute_time_diff(replaced_suggestion_human)
display(diff_human.describe())

display("Time to reaction (REPLACE) created by AI:")
diff_ai = compute_time_diff(replaced_suggestion_ai)
display(diff_ai.describe())

display("Time to reaction (REPLACE) created by Human, SELF reacted:")
diff_human_self = compute_time_diff(replaced_suggestion_human_creator)
display(diff_human_self.describe())

display("Time to reaction (REPLACE) created by Human, OTHERS reacted:")
diff_human_others = compute_time_diff(replaced_suggestion_human_others)
display(diff_human_others.describe())

display("Replaced suggestions:", replaced_suggestions["id"].count())
display("Replaced by creator:", replaced_suggestion_by_creator["id"].count())
display(replaced_suggestion_by_creator[["user_id", "card.data.user.id"]])
display("Replaced by others, total:", replaced_suggestion_by_others["id"].count())
display("Replaced by others, creator was human: ", replaced_suggestion_by_others[replaced_suggestion_by_others["card.data.user.id"] != "ai-author"]["id"].count())
display("Replaced by others, creator was AI: ", replaced_suggestion_by_others[replaced_suggestion_by_others["card.data.user.id"] == "ai-author"]["id"].count())

display(replaced_suggestion_by_others[["user_id", "card.data.user.id"]])

Index(['id', 'time', 'participant_id', 'data', 'event', 'event_uuid',
       'participant_token', 'data_parsed', 'text', 'card_id', 'user_id',
       'username', 'document_id', 'marked_text', 'refined_by_user',
       'participant_token', 'card.id', 'card.data.date', 'card.data.text',
       'card.data.user.id', 'card.data.user.tag', 'card.data.user.name',
       'card.data.user.type', 'card.data.user.online',
       'card.data.user.picture', 'card.data.user.author_id', 'card.mode',
       'card.type', 'card.state', 'card.taskId', 'card.typing', 'card.history',
       'card.replies', 'card.suggestion', 'reply.id',
       'reply.data.ai.data.prediction', 'reply.data.text', 'reply.data.time',
       'reply.data.user.id', 'reply.data.user.tag', 'reply.data.user.name',
       'reply.data.user.type', 'reply.data.user.online',
       'reply.data.user.picture', 'reply.data.user.author_id', 'reply.state',
       'reply.history', 'reply.replies', 'card.source', 'card.taskTitle',
       'card.ov

,id,time,participant_id,data,event,event_uuid,participant_token,data_parsed,text,card_id,...,reply.source,card.likeStatus,card.overlapTasks.688a6a4487cf5fec60dc80a3,card.overlapTasks.688bdf6c87cf5fec60dc80db,card.overlapTasks.688bde8287cf5fec60dc80d7,card_type,suggestion_id,card.ai.function,card.ai.prediction,card.isLoading
0,277118.0,2025-03-10 08:17:57.995,1380,"{""card"": {""id"": ""RZlhR-MXVJyAFFofc_9EU"", ""data...",SUGGESTION_TAKE_OVER,599cc0c5-47b8-4529-a1de-edcca45435fa,ecc027d0-42cd-42a1-bc2e-990b5516565b,"{'card': {'id': 'RZlhR-MXVJyAFFofc_9EU', 'data...",Digital technology has significantly impacted ...,RZlhR-MXVJyAFFofc_9EU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,277169.0,2025-03-10 08:21:18.547,1380,"{""card"": {""id"": ""LvLH98EUz1PZD85uJAEgz"", ""data...",SUGGESTION_TAKE_OVER,e4500cca-c7de-4b54-86e0-343234e5055e,ecc027d0-42cd-42a1-bc2e-990b5516565b,"{'card': {'id': 'LvLH98EUz1PZD85uJAEgz', 'data...",The primary objective of this document is to c...,LvLH98EUz1PZD85uJAEgz,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,277192.0,2025-03-10 08:22:29.447,1380,"{""card"": {""id"": ""6PoKqRSqm5nBKdEJnF7KG"", ""data...",SUGGESTION_TAKE_OVER,f702302b-7398-4df1-8170-9293aa188ba3,ecc027d0-42cd-42a1-bc2e-990b5516565b,"{'card': {'id': '6PoKqRSqm5nBKdEJnF7KG', 'data...",The goal of this document is to discuss and an...,6PoKqRSqm5nBKdEJnF7KG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,277364.0,2025-03-10 08:33:32.970,1380,"{""card"": {""id"": ""mKJ7hgCmK8cXI1G8P2fjG"", ""data...",SUGGESTION_TAKE_OVER,f94ed46f-223e-4809-9fb4-134bbc8f76e0,ecc027d0-42cd-42a1-bc2e-990b5516565b,"{'card': {'id': 'mKJ7hgCmK8cXI1G8P2fjG', 'data...",Digital technology has profoundly influenced t...,mKJ7hgCmK8cXI1G8P2fjG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,277488.0,2025-03-10 08:39:51.021,1379,"{""card"": {""id"": ""18-cROhkXZwVmH9qkyXtx"", ""data...",SUGGESTION_TAKE_OVER,d725ee98-357d-4b31-b54f-ece16b290b14,6573f86e-0207-4286-b036-a83f25a630c3,"{'card': {'id': '18-cROhkXZwVmH9qkyXtx', 'data...",The aim of this document is to shed light on t...,18-cROhkXZwVmH9qkyXtx,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,277743.0,2025-03-10 13:30:29.229,1374,"{""card"": {""id"": ""Ug9mUHiKmECYz8YUVJpU9"", ""data...",SUGGESTION_TAKE_OVER,f7da0f35-d796-4387-81c7-0b78aab8b476,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,"{'card': {'id': 'Ug9mUHiKmECYz8YUVJpU9', 'data...",Advancements in Neuro-AI technology not only e...,Ug9mUHiKmECYz8YUVJpU9,...,remote,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,277830.0,2025-03-10 14:27:48.389,1374,"{""card"": {""id"": ""SuRM3VccpleBCCxUJqLEG"", ""data...",SUGGESTION_TAKE_OVER,8cc7c3e5-11b4-47c3-8ff0-5e5a215aad49,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,"{'card': {'id': 'SuRM3VccpleBCCxUJqLEG', 'data...",The fear of job displacement due to advancing ...,SuRM3VccpleBCCxUJqLEG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,278045.0,2025-03-10 21:17:07.658,1374,"{""card"": {""id"": ""JSLbbwvrY1E39NJ-h3vEH"", ""data...",SUGGESTION_TAKE_OVER,4fe69e36-c348-456e-952c-a2395edfe6e8,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,"{'card': {'id': 'JSLbbwvrY1E39NJ-h3vEH', 'data...",The continuous evolution of AI triggers ongoin...,JSLbbwvrY1E39NJ-h3vEH,...,remote,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,278603.0,2025-03-17 14:31:26.182,1388,"{""card"": {""id"": ""V689EDFNYJpvLTVsNw3Vk"", ""data...",SUGGESTION_TAKE_OVER,3f46b21c-dfe3-4ed7-b511-bf938d4df8d2,3b49f849-3081-4478-81ce-d65450662c47,"{'card': {'id': 'V689EDFNYJpvLTVsNw3Vk', 'data...",The selected text could benefit from being bro...,V689EDFNYJpvLTVsNw3Vk,...,remote,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,278607.0,2025-03-17 14:31:29.877,1388,"{""card"": {""id"": ""eFSe0lhwKVhb2NIHTc8p5"", ""data...",SUGGESTION_TAKE_OVER,939fb17a-957e-409f-b8a5-84ecaa682a32,3b49f849-3081-4478-81ce-d65450662c47,"{'card': {'id': 'eFSe0lhwKVhb2NIHTc8p5', 'data...",The selected text is structurally sound but ca...,eFSe0lhwKVhb2NIHTc8p5,...,remote,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


'Time to reaction (REPLACE) by CREATOR:'

count     20.000000
mean      55.631800
std       46.656408
min        4.924000
25%       25.068500
50%       44.856500
75%       71.045500
max      168.523000
dtype: float64

'Time to reaction (REPLACE) by OTHERS:'

...

'Replaced by others, creator was human: '

22

'Replaced by others, creator was AI: '

8

,user_id,card.data.user.id
5,67cdfa30285fac3f07bcf5cc,67cdfc85285fac3f07bcf5cf
7,67cdfa30285fac3f07bcf5cc,67cdfc85285fac3f07bcf5cf
8,67d82dc4edfe680c1b0f1642,67d82f6dedfe680c1b0f164a
9,67d82dc4edfe680c1b0f1642,67d82f6dedfe680c1b0f164a
10,67d82dc4edfe680c1b0f1642,67d82f6dedfe680c1b0f164a
11,67d82dc4edfe680c1b0f1642,67d82f6dedfe680c1b0f164a
12,67d82dc4edfe680c1b0f1642,67d82f6dedfe680c1b0f164a
13,67d82dc4edfe680c1b0f1642,67d82f6dedfe680c1b0f164a
14,67d82e0aedfe680c1b0f1643,ai-author
15,67d82dc4edfe680c1b0f1642,ai-author


In [51]:
# number of c2c suggestions
c2c_suggestions = parse_generic_event(finished_participants, "SUGGESTION_COPY_TO_CLIPBOARD")
c2c_suggestions = transform_reaction_times(c2c_suggestions)

display(c2c_suggestions.columns)
display(c2c_suggestions)

c2ced_suggestion_by_creator = c2c_suggestions[c2c_suggestions["user_id"] == c2c_suggestions["card.data.user.id"]]
c2ced_suggestion_by_others = c2c_suggestions[c2c_suggestions["user_id"] != c2c_suggestions["card.data.user.id"]]

c2ced_suggestion_human = c2c_suggestions[c2c_suggestions["card.data.user.id"] != "ai-author"]
c2ced_suggestion_ai = c2c_suggestions[c2c_suggestions["card.data.user.id"] == "ai-author"]

c2ced_suggestion_human_creator = c2ced_suggestion_human[c2ced_suggestion_human["user_id"] == c2ced_suggestion_human["card.data.user.id"]]
c2ced_suggestion_human_others = c2ced_suggestion_human[c2ced_suggestion_human["user_id"] != c2ced_suggestion_human["card.data.user.id"]]

display("Time to reaction (C2C) by CREATOR:")
diff_creator = compute_time_diff(c2ced_suggestion_by_creator)
display(diff_creator.describe())

display("Time to reaction (C2C) by OTHERS:")
diff_others = compute_time_diff(c2ced_suggestion_by_others)
display(diff_others.describe())

display("Time to reaction (C2C) COMBINED:")
diff = pd.concat([diff_creator, diff_others])
display(diff.describe())

display("Time to reaction (C2C) created by HUMAN:")
diff_human = compute_time_diff(c2ced_suggestion_human)
display(diff_human.describe())

display("Time to reaction (C2C) created by AI:")
diff_ai = compute_time_diff(c2ced_suggestion_ai)
display(diff_ai.describe())

display("Time to reaction (C2c) created by Human, SELF reacted:")
diff_human_self = compute_time_diff(c2ced_suggestion_human_creator)
display(diff_human_self.describe())

display("Time to reaction (C2C) created by Human, OTHERS reacted:")
diff_human_others = compute_time_diff(c2ced_suggestion_human_others)
display(diff_human_others.describe())


display("C2Ced suggestions:", c2c_suggestions["id"].count())
display("C2Ced by creator:", c2ced_suggestion_by_creator["id"].count())
display(c2ced_suggestion_by_creator[["user_id", "card.data.user.id"]])
display("C2Ced by others, total:", c2ced_suggestion_by_others["id"].count())
display("C2Ced by others, creator was human: ", c2ced_suggestion_by_others[c2ced_suggestion_by_others["card.data.user.id"] != "ai-author"]["id"].count())
display("C2Ced by others, creator was AI: ", c2ced_suggestion_by_others[c2ced_suggestion_by_others["card.data.user.id"] == "ai-author"]["id"].count())

display(c2ced_suggestion_by_others[["user_id", "card.data.user.id"]])


Index(['id', 'time', 'participant_id', 'data', 'event', 'event_uuid',
       'participant_token', 'data_parsed', 'text', 'card_id', 'user_id',
       'username', 'document_id', 'marked_text', 'refined_by_user',
       'participant_token', 'card.id', 'card.data.date', 'card.data.text',
       'card.data.user.id', 'card.data.user.tag', 'card.data.user.name',
       'card.data.user.type', 'card.data.user.online',
       'card.data.user.picture', 'card.data.user.author_id', 'card.mode',
       'card.type', 'card.state', 'card.typing', 'card.history',
       'card.replies', 'card.likeStatus', 'card.suggestion', 'reply.id',
       'reply.data.ai.data.prediction', 'reply.data.text', 'reply.data.time',
       'reply.data.user.id', 'reply.data.user.tag', 'reply.data.user.name',
       'reply.data.user.type', 'reply.data.user.online',
       'reply.data.user.picture', 'reply.data.user.author_id', 'reply.state',
       'reply.history', 'reply.replies', 'card.source', 'card.taskId',
       'card.t

,id,time,participant_id,data,event,event_uuid,participant_token,data_parsed,text,card_id,...,reply.replies,card.source,card.taskId,card.taskTitle,reply.source,card_type,suggestion_id,card.ai.prediction,card.isLoading,card.overlapTasks.689c51f2e90ffc190a96c297
0,276852.0,2025-03-10 08:08:02.009,1379,"{""card"": {""id"": ""XQt5XrkOapXdkdaV-J3Cn"", ""data...",SUGGESTION_COPY_TO_CLIPBOARD,a80f2e5b-81a7-4b8c-9497-c88218388695,6573f86e-0207-4286-b036-a83f25a630c3,"{'card': {'id': 'XQt5XrkOapXdkdaV-J3Cn', 'data...","Sure, for your essay ""Health and Wellness in t...",XQt5XrkOapXdkdaV-J3Cn,...,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,277068.0,2025-03-10 08:16:31.871,1379,"{""card"": {""id"": ""mHIkbNGUYqepeZ-_NGSVN"", ""data...",SUGGESTION_COPY_TO_CLIPBOARD,b6380974-cdd3-4977-8484-09a1a7eb8ca5,6573f86e-0207-4286-b036-a83f25a630c3,"{'card': {'id': 'mHIkbNGUYqepeZ-_NGSVN', 'data...",The goal of the document is to define health a...,mHIkbNGUYqepeZ-_NGSVN,...,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,277290.0,2025-03-10 08:26:40.971,1379,"{""card"": {""id"": ""nOqZNstq0_qlc9q0-0c4X"", ""data...",SUGGESTION_COPY_TO_CLIPBOARD,06582656-0212-419c-8c38-51516a690fbf,6573f86e-0207-4286-b036-a83f25a630c3,"{'card': {'id': 'nOqZNstq0_qlc9q0-0c4X', 'data...","Sure, I can provide some statistics related to...",nOqZNstq0_qlc9q0-0c4X,...,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,277314.0,2025-03-10 08:30:45.029,1379,"{""card"": {""id"": ""HDRrJll97j9BSVFDgbXsM"", ""data...",SUGGESTION_COPY_TO_CLIPBOARD,8c378c8c-0b7c-4916-a5da-3d2d02fa585e,6573f86e-0207-4286-b036-a83f25a630c3,"{'card': {'id': 'HDRrJll97j9BSVFDgbXsM', 'data...",The goal of the document is to analyze and dis...,HDRrJll97j9BSVFDgbXsM,...,[],remote,67cea2a4285fac3f07bcf5f8,Conclusion Writing Task,remote,NaN,NaN,NaN,NaN,NaN
4,277483.0,2025-03-10 08:39:33.977,1379,"{""card"": {""id"": ""18-cROhkXZwVmH9qkyXtx"", ""data...",SUGGESTION_COPY_TO_CLIPBOARD,91c348d1-14ee-4fcd-b945-7661a8a06cbe,6573f86e-0207-4286-b036-a83f25a630c3,"{'card': {'id': '18-cROhkXZwVmH9qkyXtx', 'data...",The aim of this document is to shed light on t...,18-cROhkXZwVmH9qkyXtx,...,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,278623.0,2025-03-17 14:31:35.531,1389,"{""card"": {""id"": ""2XJShiticuPh5fZaWgaio"", ""data...",SUGGESTION_COPY_TO_CLIPBOARD,7b817cc4-be76-487c-9bee-cc4f5675e1ea,ee4fc4b9-7b40-4acc-8299-bf6e2da559f9,"{'card': {'id': '2XJShiticuPh5fZaWgaio', 'data...","The selected text is clear, but it can be rest...",2XJShiticuPh5fZaWgaio,...,[],remote,67d82f87edfe680c1b0f164b,Text Error Check,remote,NaN,NaN,NaN,NaN,NaN
6,280287.0,2025-07-28 14:56:15.504,2456,"{""card"": {""id"": ""iPa488CcUmXhp1MFqZqIt"", ""data...",SUGGESTION_COPY_TO_CLIPBOARD,f7c95511-aaaf-4ec7-b5f5-77d935801cce,ee84f721-81d3-4914-b1a4-09de1502b0bc,"{'card': {'id': 'iPa488CcUmXhp1MFqZqIt', 'data...",Here’s a revised version of the selected secti...,iPa488CcUmXhp1MFqZqIt,...,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,281067.0,2025-07-29 13:23:03.445,2459,"{""card"": {""id"": ""f9aixUpkOGZUJTQrxJuDS"", ""data...",SUGGESTION_COPY_TO_CLIPBOARD,55f24bdf-d7a4-4bfd-84ce-a6a4be5ce69f,a1f051f1-f2f3-45e1-9274-7deb05f97604,"{'card': {'id': 'f9aixUpkOGZUJTQrxJuDS', 'data...",Here’s a purchase list based on the selected t...,f9aixUpkOGZUJTQrxJuDS,...,[],NaN,6888c9ec87cf5fec60dc8082,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,282820.0,2025-08-14 08:35:16.811,2468,"{""card"": {""ai"": {""prediction"": ""\n\nClassical ...",SUGGESTION_COPY_TO_CLIPBOARD,aaf24833-8fa8-4445-b456-722d2216d414,b7d11063-8fd9-4d87-b9c4-afa680ba7235,{'card': {'ai': {'prediction': ' Classical me...,\n\nClassical methods for Information Extracti...,DJnnMXYQSLxCEHO0qKj2G,...,NaN,NaN,NaN,NaN,NaN,AI_EXTEND,NaN,\n\nClassical methods for Information Extracti...,False,Reference Verification
9,283098.0,2025-08-15 07:16:48.256,2468,"{""card"": {""id"": ""L-aEtzw2Ia9S3baBPXhaL"", ""data...",SUGGESTION_COPY_TO_CLIPBOARD,e1d98268-62c3-4750-923a-1b3b42ab4f40,b7d11063-8fd9-4d87-b9c4-afa680ba7235,"{'card': {'id': 'L-aEtzw2Ia9S3

'Time to reaction (C2C) by CREATOR:'

count      7.000000
mean     134.334714
std      115.093796
min       33.821000
25%       49.973500
50%       88.080000
75%      199.884500
max      318.726000
dtype: float64

'Time to reaction (C2C) by OTHERS:'

...

'C2Ced by others, creator was human: '

5

'C2Ced by others, creator was AI: '

1

,user_id,card.data.user.id
0,67ce982f285fac3f07bcf5e7,67ce9cb4285fac3f07bcf5ec
3,67ce982f285fac3f07bcf5e7,ai-author
5,67d82e0aedfe680c1b0f1643,67d82f6dedfe680c1b0f164a
9,6898407be90ffc190a96c287,689c51e2e90ffc190a96c296
10,68a75365ecc07d2944852275,68931d9887cf5fec60dc810f
12,68a75365ecc07d2944852275,688b2aa687cf5fec60dc80b3


In [52]:
# number of c2c suggestions
all_accepts = parse_generic_event(finished_participants, ["SUGGESTION_COPY_TO_CLIPBOARD", "SUGGESTION_TAKE_OVER", "SUGGESTION_INSERT_AFTER"])
all_accepts = transform_reaction_times(all_accepts)

display(all_accepts.columns)
display(all_accepts)

all_accepts_by_creator = all_accepts[all_accepts["user_id"] == all_accepts["card.data.user.id"]]
all_accepts_by_others = all_accepts[all_accepts["user_id"] != all_accepts["card.data.user.id"]]

all_accepts_human = all_accepts[all_accepts["card.data.user.id"] != "ai-author"]
all_accepts_ai = all_accepts[all_accepts["card.data.user.id"] == "ai-author"]

all_accepts_human_creator = all_accepts_human[all_accepts_human["user_id"] == all_accepts_human["card.data.user.id"]]
all_accepts_human_others = all_accepts_human[all_accepts_human["user_id"] != all_accepts_human["card.data.user.id"]]

display("Time to reaction (ALL METHODS) by CREATOR:")
diff_creator = compute_time_diff(all_accepts_by_creator)
diff_creator = diff_creator / 60
display(diff_creator.describe())

display("Time to reaction (ALL METHODS) by OTHERS:")
diff_others = compute_time_diff(all_accepts_by_others)
diff_others = diff_others / 60
display(diff_others.describe())

display("Time to reaction (ALL METHODS) COMBINED:")
diff = pd.concat([diff_creator, diff_others])
display(diff.describe())

display("Time to reaction (ALL METHODS) created by HUMAN:")
display(all_accepts_human[["card.id","card.data.user.id", "card.data.user.name"]])
diff_human = compute_time_diff(all_accepts_human)
diff_human = diff_human / 60
display(diff_human.describe())

display("Time to reaction (ALL METHODS) created by AI:")
display(all_accepts_ai)
diff_ai = compute_time_diff(all_accepts_ai)
diff_ai = diff_ai / 60
display(diff_ai.describe())

display("Time to reaction (ALL METHODS) created by Human, SELF reacted:")
diff_human_self = compute_time_diff(all_accepts_human_creator)
diff_human_self = diff_human_self / 60
display(diff_human_self.describe())

display("Time to reaction (ALL METHODS) created by Human, OTHERS reacted:")
diff_human_others = compute_time_diff(all_accepts_human_others)
diff_human_others = diff_human_others / 60
display(diff_human_others.describe())


display("ALL METHODS suggestions:", all_accepts["id"].count())
display("ALL METHODS by creator:", all_accepts_by_creator["id"].count())
display(all_accepts_by_creator[["user_id", "card.data.user.id"]])
display("ALL METHODS by others, total:", all_accepts_by_others["id"].count())
display("ALL METHODS by others, creator was human: ", all_accepts_by_others[all_accepts_by_others["card.data.user.id"] != "ai-author"]["id"].count())
display("ALL METHODS by others, creator was AI: ", all_accepts_by_others[all_accepts_by_others["card.data.user.id"] == "ai-author"]["id"].count())

display(c2ced_suggestion_by_others[["user_id", "card.data.user.id"]])


Index(['id', 'time', 'participant_id', 'data', 'event', 'event_uuid',
       'participant_token', 'data_parsed', 'text', 'card_id', 'user_id',
       'username', 'document_id', 'marked_text', 'suggestion_id',
       'refined_by_user', 'participant_token', 'card.id', 'card.data.date',
       'card.data.text', 'card.data.user.id', 'card.data.user.tag',
       'card.data.user.name', 'card.data.user.type', 'card.data.user.online',
       'card.data.user.picture', 'card.mode', 'card.type', 'card.state',
       'card.source', 'card.taskId', 'card.typing', 'card.history',
       'card.replies', 'card.taskTitle', 'reply.id',
       'reply.data.ai.data.prediction', 'reply.data.text', 'reply.data.time',
       'reply.data.user.id', 'reply.data.user.tag', 'reply.data.user.name',
       'reply.data.user.type', 'reply.data.user.online',
       'reply.data.user.picture', 'reply.state', 'reply.source',
       'reply.history', 'reply.replies', 'card_type', 'card.ai.prediction',
       'card.data.user.

,id,time,participant_id,data,event,event_uuid,participant_token,data_parsed,text,card_id,...,card.likeStatus,card.overlapTasks.67cdfd24285fac3f07bcf5d1,card.overlapTasks.67d83435edfe680c1b0f1656,card.suggestion.id,card.suggestion.user,card.overlapTasks.688a6a4487cf5fec60dc80a3,card.overlapTasks.688bdf6c87cf5fec60dc80db,card.overlapTasks.688bde8287cf5fec60dc80d7,card.ai.function,card.overlapTasks.689c51f2e90ffc190a96c297
0,276239.0,2025-03-07 21:42:34.724,1377,"{""card"": {""id"": ""h9dJoEjrePKz1x9XOxrxx"", ""data...",SUGGESTION_INSERT_AFTER,31658bce-0fbf-4559-9541-0242b1f07006,2aac5afe-a3af-4135-a1a3-78adf87ae9d8,"{'card': {'id': 'h9dJoEjrePKz1x9XOxrxx', 'data...","In this essay, we will delve into the various ...",h9dJoEjrePKz1x9XOxrxx,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,276285.0,2025-03-07 21:44:21.621,1376,"{""card"": {""id"": ""5Plaf8nQ31ghtWn4OAFbg"", ""data...",SUGGESTION_INSERT_AFTER,b0449168-c323-4422-87a9-b21ded2e7520,a5bbafd1-4ef0-49e4-95bd-65170953633c,"{'card': {'id': '5Plaf8nQ31ghtWn4OAFbg', 'data...",The aim of this part of the document is to thr...,5Plaf8nQ31ghtWn4OAFbg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,276320.0,2025-03-07 21:45:43.489,1377,"{""card"": {""ai"": {""prediction"": "" \n\nFirst and...",SUGGESTION_INSERT_AFTER,8c4b3c43-7323-46e2-98ea-745c70df289d,2aac5afe-a3af-4135-a1a3-78adf87ae9d8,{'card': {'ai': {'prediction': ' First and f...,"\n\nFirst and foremost, AI has the ability to...",8A2-OJj7uKw6ckP7d2DVP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,276347.0,2025-03-07 21:46:31.484,1376,"{""card"": {""id"": ""T1140AO9etlFPy5sX0me-"", ""data...",SUGGESTION_INSERT_AFTER,ce9eb039-b494-481b-8dbf-5329df65e520,a5bbafd1-4ef0-49e4-95bd-65170953633c,"{'card': {'id': 'T1140AO9etlFPy5sX0me-', 'data...",The objective of the document is to provide a ...,T1140AO9etlFPy5sX0me-,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,276387.0,2025-03-07 21:49:31.354,1376,"{""card"": {""id"": ""EiPiEDL-QXWG5qiQOj9tm"", ""data...",SUGGESTION_INSERT_AFTER,07f107d3-e7ba-4491-9556-25bd48034de4,a5bbafd1-4ef0-49e4-95bd-65170953633c,"{'card': {'id': 'EiPiEDL-QXWG5qiQOj9tm', 'data...",Document Goal: \n\nThe primary objective of th...,EiPiEDL-QXWG5qiQOj9tm,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,284423.0,2025-08-22 17:18:30.274,2475,"{""card"": {""id"": ""0_vn2aveJwiAmNBnn1HPB"", ""data...",SUGGESTION_COPY_TO_CLIPBOARD,8d84e06f-dba3-45f6-8481-c2ebe1cfa28b,5d9124de-67e0-4bd1-80e6-ec33aa775d1a,"{'card': {'id': '0_vn2aveJwiAmNBnn1HPB', 'data...",Here is a detailed lesson plan for a graduate ...,0_vn2aveJwiAmNBnn1HPB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
131,284430.0,2025-08-22 17:19:55.096,2475,"{""card"": {""id"": ""0_vn2aveJwiAmNBnn1HPB"", ""data...",SUGGESTION_INSERT_AFTER,c9d50ccb-5813-4f22-9af2-6e72fc3450f8,5d9124de-67e0-4bd1-80e6-ec33aa775d1a,"{'card': {'id': '0_vn2aveJwiAmNBnn1HPB', 'data...",Here is a detailed lesson plan for a graduate ...,0_vn2aveJwiAmNBnn1HPB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132,284468.0,2025-08-22 17:34:07.498,2475,"{""card"": {""id"": ""T6dovEXM5kHzPLUczuYjE"", ""data...",SUGGESTION_COPY_TO_CLIPBOARD,2d544194-aa71-4fe0-8737-13e246c4bba3,5d9124de-67e0-4bd1-80e6-ec33aa775d1a,"{'card': {'id': 'T6dovEXM5kHzPLUczuYjE', 'data...",The literature references provided focus on re...,T6dovEXM5kHzPLUczuYjE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
133,284470.0,2025-08-22 17:36:11.852,2475,"{""card"": {""id"": ""T6dovEXM5kHzPLUczuYjE"", ""data...",SUGGESTION_INSERT_AFTER,9c465731-1d58-4881-8f6f-6169ff1f22fa,5d9124de-67e0-4bd1-80e6-ec33aa775d1a,"{'card': {'id': 'T6dovEXM5kHzPLUczuYjE', 'data...",The literature references provided focus on re...,T6dovEXM5kHzPLUczuYjE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


'Time to reaction (ALL METHODS) by CREATOR:'

count    73.000000
mean      1.082009
std       1.391608
min       0.068883
25%       0.308400
50%       0.630250
75%       1.300200
max       8.128517
dtype: float64

'Time to reaction (ALL METHODS) by OTHERS:'

...

'ALL METHODS by others, creator was human: '

46

'ALL METHODS by others, creator was AI: '

16

,user_id,card.data.user.id
0,67ce982f285fac3f07bcf5e7,67ce9cb4285fac3f07bcf5ec
3,67ce982f285fac3f07bcf5e7,ai-author
5,67d82e0aedfe680c1b0f1643,67d82f6dedfe680c1b0f164a
9,6898407be90ffc190a96c287,689c51e2e90ffc190a96c296
10,68a75365ecc07d2944852275,68931d9887cf5fec60dc810f
12,68a75365ecc07d2944852275,688b2aa687cf5fec60dc80b3


In [53]:
# number of inserted suggestions
finished_participants[finished_participants["event"] == "SUGGESTION_INSERT_AFTER"]["data"].count()

72

In [54]:
inserted_after_df = finished_participants[finished_participants["event"] == "SUGGESTION_INSERT_AFTER"]

# collect cards ids which were used in text. This set will be extended further to take into account
# SUGGESTION_INSERT_AFTER, SUGGESTION_TAKE_OVER and those cards, whose text could be found in text document 
# (I consider that the card text was used because it's suggestion is in text)
card_ids = set()

for idx, row in inserted_after_df.iterrows():
    info = json.loads(row["data"])

    if 'card' in info:
        card_ids.add(info["card"]["id"])

    
print(len(card_ids))

71


In [55]:
take_over_df = finished_participants[finished_participants["event"] == "SUGGESTION_TAKE_OVER"]

for idx, row in take_over_df.iterrows():
    info = json.loads(row["data"])

    if 'card' in info:
        card_ids.add(info["card"]["id"])

    

print(len(card_ids))

121


In [56]:
remove_events_starting_from = [
    (1374, '2025-03-10 14:44:50.308'),
    #(1374, '2025-03-09 22:18:40.957'), # last comment approve
    #(1378, '2025-03-08 20:30:28.499'),
    (1388, '2025-03-17 15:12:56.935'), # last suggestion view detail
    (1389, '2025-03-17 15:00:33.242'),
    (1387, '2025-03-15 20:09:12.496'), # last comment approve
    (1390, '2025-03-15 20:09:12.496'), # this participant reported that they did not collaborate
    (2456, '2025-07-28 18:04:03.337'), # last task view close
    (2453, '2025-07-21 09:45:35.067'), # last agent view close 
    (2455, '2025-07-23 13:53:19.984'), # last suggestion view detail
    (2461, '2025-07-31 21:50:47.107'), # last suggestion take over
    (2464, '2025-07-30 19:28:28.289'), # last comment approve
    (2457, '2025-07-31 15:31:45.052'), # last reply post
    (2459, '2025-07-29 13:23:03.445'), # last suggestion c2c
    (2465, '2025-07-31 21:20:40.047'), # last suggestion view detail
    (2467, '2025-08-20 16:21:49.455'), # last reply post
    (2476, '2025-08-20 16:15:56.221') # last comment related event
]


In [57]:
user_lookup = {
    '67ce982f285fac3f07bcf5e7': [1, 1379],
    '67ce9cb4285fac3f07bcf5ec': [1, 1380],
    '67cb6627285fac3f07bcf59b': [2, 1376],
    '67cb6623285fac3f07bcf59a': [2, 1377],
    '67cc9f69285fac3f07bcf5c8': [3, 1378],
    '67cdfa30285fac3f07bcf5cc': [3, 1374],
    '67bc761a7a670b68746905a0': [4, 1383],
    '67d444dd0b9539e69a3a86bb': [4, 1384],
    '67bc43607a670b6874690577': [4, 1385],
    '67d5d8120b9539e69a3a86d7': [5, 1387],
    '67e18e95edfe680c1b0f169f': [5, 1390],
    '67d82dc4edfe680c1b0f1642': [6, 1388],
    '67d82e0aedfe680c1b0f1643': [6, 1389],
    '68835b0587cf5fec60dc8020': [7, 2456],
    '6887b97087cf5fec60dc8060': [7, 2460],
    '687de5d187cf5fec60dc7fef': [8, 2453],
    '6880e0f187cf5fec60dc7ffe': [8, 2455],
    '6888eec887cf5fec60dc8085': [9, 2461],
    '688a52ad87cf5fec60dc8096': [9, 2464],
    '688b1c7887cf5fec60dc80b0': [9, 2457],
    '6888bbce87cf5fec60dc8072': [10, 2459],
    '688bdb6387cf5fec60dc80d0': [10, 2465],
    '6889e71f87cf5fec60dc8094': [11, 2463],
    '68a75365ecc07d2944852275': [11, 2475],
    '6898407be90ffc190a96c287': [12, 2468],
    '689c3c25e90ffc190a96c28b': [12, 2469],
    '68a5a0bde90ffc190a96c2ff': [13, 2467],
    '68a59766e90ffc190a96c2f4': [13, 2476],
    '689e2181e90ffc190a96c2df': [14, 2470],
    '689e22d5e90ffc190a96c2e0': [14, 2471],
}

[Q]Number of replies with specific patterns
[O]Average number of comments resulting from running a task

In [58]:
docs = [
    "67ce983b285fac3f07bcf5e8", 
    "67c97527285fac3f07bcf587",
    "67c88f88285fac3f07bcf585", 
    "67d43b390b9539e69a3a86b9",
    "67d5d8160b9539e69a3a86d8",
    "67d08e87285fac3f07bcf64b",
    "68833fd687cf5fec60dc801d",
    "6879e8fd87cf5fec60dc7fe1",
    "688364bf87cf5fec60dc8026",
    "6877c26e87cf5fec60dc7fda",
    "6889e57087cf5fec60dc8092",
    "688c6add87cf5fec60dc80ef",
    "68907a6e87cf5fec60dc8107",
    "6896071ee90ffc190a96c285"
]

participants_groups = {
    "1": {"users": [1379, 1380], "user_ids": ['67ce982f285fac3f07bcf5e7', '67ce9cb4285fac3f07bcf5ec'], "documents": ["67ce983b285fac3f07bcf5e8"]},
    "2": {"users": [1376, 1377], "user_ids": ['67cb6627285fac3f07bcf59b', '67cb6623285fac3f07bcf59a'], "documents": ["67c97527285fac3f07bcf587"]},
    "3": {"users": [1378, 1374], "user_ids": ['67cc9f69285fac3f07bcf5c8', '67cdfa30285fac3f07bcf5cc'], "documents": ["67c88f88285fac3f07bcf585"]},
    "4": {"users": [1383, 1384, 1385], "user_ids": ['67bc761a7a670b68746905a0', '67d444dd0b9539e69a3a86bb', '67bc43607a670b6874690577'], "documents": ["67d43b390b9539e69a3a86b9"]},
    "5": {"users": [1387, 1390], "user_ids": ['67d5d8120b9539e69a3a86d7', '67e18e95edfe680c1b0f169f'], "documents": ["67d5d8160b9539e69a3a86d8"]},
    "6": {"users": [1388, 1389], "user_ids": ['67d82dc4edfe680c1b0f1642', '67d82e0aedfe680c1b0f1643'], "documents": ["67d08e87285fac3f07bcf64b"]},
    "7": {"users": [2456, 2460], "user_ids": ['68835b0587cf5fec60dc8020', '6887b97087cf5fec60dc8060'], "documents": ["68833fd687cf5fec60dc801d"]},
    "8": {"users": [2453, 2455], "user_ids": ['687de5d187cf5fec60dc7fef', '6880e0f187cf5fec60dc7ffe'], "documents": ["6879e8fd87cf5fec60dc7fe1"]},
    "9": {"users": [2461, 2464, 2457], "user_ids": ['6888eec887cf5fec60dc8085', '688a52ad87cf5fec60dc8096', '688b1c7887cf5fec60dc80b0'], "documents": ["688364bf87cf5fec60dc8026"]},
    "10": {"users": [2459, 2465], "user_ids": ['6888bbce87cf5fec60dc8072', '688bdb6387cf5fec60dc80d0'], "documents": ["6877c26e87cf5fec60dc7fda"]},
    "11": {"users": [2463, 2475], "user_ids": ['6889e71f87cf5fec60dc8094', '68a75365ecc07d2944852275'], "documents": ["6889e57087cf5fec60dc8092"]},
    "12": {"users": [2468, 2469], "user_ids": ['6898407be90ffc190a96c287', '689c3c25e90ffc190a96c28b'], "documents": ["688c6add87cf5fec60dc80ef"]},
   # "13": {"users": [2467, 2476], "user_ids": ['68a5a0bde90ffc190a96c2ff', '68a59766e90ffc190a96c2f4'], "documents": ["68907a6e87cf5fec60dc8107"]},
    "14": {"users": [2470, 2471], "user_ids": ['689e2181e90ffc190a96c2df', '689e22d5e90ffc190a96c2e0'], "documents": ["6896071ee90ffc190a96c285"]},
} 


# read document collection
data = documents_df

# number of card with type of comment = number of chats started
total_chats = 0
total_chats_before = 0
total_replies = 0

comments_without_text = 0

doc_stat = dict()
max_stat = {doc: 0 for doc in docs}

# count number of liked and disliked cards
like_status = 0
dislike_status = 0

# number of cards for tasks that run on the whole document
task_comment_count = 0
task_comment_by_users_count= 0
task_comment_by_ai = 0

# number of messages that mention agent
ai_mentions = 0
# number of cards that mention agent in the first message
ai_start_text_mentions = 0
ai_mentions_replies = 0

# number of messages sent by user
user_input_count = 0
user_created_comments = 0
user_replies = 0


# discussions
only_a_comment = 0
human_human_discussions = 0
human_ai_discussions = 0
ai_human_discussions = 0
human_ai_nego_discussions = 0
human_team_ai_discussions = 0

human_ai_discussions_list = []

user_created_comments_list = []
task_id_list = []
ad_hoc_comment_tasks = []
ad_hoc_reply_tasks = []

comment_threads_short = []
comment_threads_long = []

#decisions = []

#number of messgaes sent by AI
ai_created_comments = 0
ai_replies = 0
ai_created_comment_no_task = 0

# docs with ids as required
data = data[data["_id"].isin(docs)]

# 67c88f88285fac3f07bcf585

# for each check
for idx, row in data.iterrows():
        
    # get cards

        # check and replace "helvetica neue" inline css style in cards data, since it breaks the json parsing of cards for group 1
        if (row["cards"].find('"Helvetica Neue",') != -1):
            print("Helvetica neue in text:", idx)
        row["cards"] = row["cards"].replace('"Helvetica Neue",', '')

        info = json.loads(row["cards"])
        
        #display(idx)
        #print(row["cards"])
        
        #total_chats += len(info)
        total_chats_before += len(info)
        
        # for each chat get the amount of replies
        n = 0
        
        for chat in info.values():
            
            #check if user has time constraint
            #user[1] retrieves the user id from lookup struct

            # get time constraint from remove_events_starting_from struct
            # first get user id by mongo db user id
            time_constraint = None
            if chat['data']['user']['type'] != "ai":
                
                # skip ghost user (user that has been used after the study)
                if chat['data']['user']['id'] == "67d5bdff0b9539e69a3a86c9":
                    continue
                
                user = user_lookup[chat['data']['user']['id']]
                if user:
                    for tc in remove_events_starting_from:
                        if user[1] == tc[0]:
                           time_constraint = datetime.strptime(tc[1], '%Y-%m-%d %H:%M:%S.%f')

            if chat["type"] == "comment":
                
                chat_date = datetime.utcfromtimestamp(chat['data']['date'] / 1000)
                print(chat_date, time_constraint)
                if time_constraint and chat_date > time_constraint:
                    print("removed because of time_constraint")
                    print(time_constraint)
                    print(chat["data"]["user"]["id"])
                    continue
                
                # only add comment cards to chat count
                total_chats += 1
                
                x = len(chat["replies"])
                max_stat[row["_id"]] = max(max_stat[row["_id"]], x)
    
                if "likeStatus" in chat:
                    if chat["likeStatus"] == "like":
                        like_status += 1
                    elif chat["likeStatus"] == "dislike":
                        dislike_status += 1
                
                ai_comment = False
                user_comment = False
                user_task_comment = False
                comment_user_id = None
                reply_user_ids = []

                if "data" in chat:
                    if "text" in chat["data"]:

                        if "user" in chat["data"]:
                            comment_user_id = chat['data']['user']['id']
                        
                        if "taskId" in chat:
                        #if chat["data"]["text"].startswith("Run task"):
                            task_id_list.append(chat["taskId"])
                            task_comment_count += 1
                            if chat["data"]["user"]["type"] != "ai":
                                task_comment_by_users_count += 1
                                user_task_comment = True
                            else: 
                                task_comment_by_ai += 1 
                                
                                ai_created_comments += 1
                                ai_comment = True
                        else:
                            if chat["data"]["user"]["type"] != "ai":
                                user_input_count += 1
                                
                                user_created_comments_list.append(chat)
                                user_created_comments += 1
                                user_comment = True
                            else:   
                                ai_created_comment_no_task += 1
                                                              
                            if "@ai" in chat["data"]["text"]:
                                ai_start_text_mentions += 1
                                ai_mentions += 1
                                display(chat)
                                ad_hoc_comment_tasks.append({"text":chat["data"]["text"], "user_id": chat['data']['user']['id'], "comment_id": chat['id'], "assignee_id": chat["replies"][0]["data"]["user"]["id"], "assignee_name": chat["replies"][0]["data"]["user"]["name"]})
                    else:
                        comments_without_text += 1

                if "replies" in chat:
                
                    # only count replies when it exists, e.g. for comments
                    n += x
                    
                    ai_reply = False
                    user_reply = False
                    
                    chat_text = "" if not "text" in chat["data"] else chat["data"]["text"]
                
                    comment_thread = [{"comment": chat_text, "user": chat["data"]["user"]["id"], "type": chat["data"]["user"]["type"], "name": chat["data"]["user"]["name"]}]
                    for idx, reply in enumerate(chat["replies"]):
                        # print(reply)
                        if "data" in reply and "user" in reply["data"] and "type" in reply["data"]["user"]:
                            
                            # skip ghost user (user that has been used after the study)
                            if reply['data']['user']['id'] == "67d5bdff0b9539e69a3a86c9":
                                continue
                            
                            if reply["data"]["user"]["type"] == "ai":
                                comment_thread.append({"reply": reply["data"]["ai"]["data"]["prediction"], "user": reply["data"]["user"]["id"], "ai": reply["data"]["user"]["name"]})
                                ai_replies += 1
                                ai_reply = True
                            else:
                                comment_thread.append({"reply": reply["data"]["text"], "user": reply["data"]["user"]["id"], "human": reply["data"]["user"]["name"]})
                                user_replies += 1
                                user_input_count += 1
                                user_reply = True
                                reply_user_ids.append(reply['data']['user']['id'])
        
                        if "data" in reply and "text" in reply["data"]:
                            if "@ai" in reply["data"]["text"]:
                                ai_mentions += 1
                                ai_mentions_replies += 1

                                assignee_id = None
                                assignee_name = None
                                if idx + 1 < len(chat["replies"]):
                                    assignee_id = chat["replies"][idx + 1]["data"]["user"]["id"] 
                                    assignee_name = chat["replies"][idx + 1]["data"]["user"]["name"] 
                
                                ad_hoc_reply_tasks.append({"text":reply["data"]["text"], "user_id": reply['data']['user']['id'], "comment_id": chat['id'], "user_name": reply['data']['user']['id'], "assignee_id": assignee_id, "assignee_name": assignee_name})
                                
                discussion_type = ""
                filtered_reply_user_ids = [user_id for user_id in reply_user_ids if user_id != comment_user_id]
                if user_comment and ai_reply and not user_reply:
                    human_ai_discussions += 1
                    human_ai_discussions_list.append({"chat": chat, "text":chat["data"]["text"], "user_id": chat['data']['user']['id'], "comment_id": chat['id']})
                    discussion_type = "HUMAN_AI_DISCUSSION"
                if user_comment and ai_reply and user_reply and len(filtered_reply_user_ids) == 0: # replies only by comment user
                    human_ai_nego_discussions += 1
                    discussion_type = "HUMAN_AI_NEGOTIATION_DISCUSSION"
                if user_comment and ai_reply and user_reply and len(filtered_reply_user_ids) > 0: # also replies by other users
                    human_team_ai_discussions += 1
                    discussion_type = "HUMAN_TEAM_AI_DISCUSSION"
                if user_comment and user_reply and not ai_reply and len(filtered_reply_user_ids) > 0:
                    human_human_discussions += 1
                    discussion_type = "HUMAN_HUMAN_DISCUSSION"
                if ai_comment and user_reply:
                    ai_human_discussions += 1
                    discussion_type = "AI_HUMAN_DISCUSSION"
                if user_comment and not ai_reply and not user_reply:
                    only_a_comment += 1
                    discussion_type = "ONLY_COMMENT"
                        
                if len(comment_thread) > 2: #only extract threads that have 2 or more replies
                    
                    if user_task_comment and ai_reply and not user_reply:
                        discussion_type = "TASK_HUMAN_AI_DISCUSSION"
                    if user_task_comment and ai_reply and user_reply and len(filtered_reply_user_ids) == 0: # replies only by comment user
                        discussion_type = "TASK_HUMAN_AI_NEGOTIATION_DISCUSSION"
                    if user_task_comment and ai_reply and user_reply and len(filtered_reply_user_ids) > 0: # also replies by other users
                        discussion_type = "TASK_HUMAN_TEAM_AI_DISCUSSION"
                    if user_task_comment and user_reply and not ai_reply and len(filtered_reply_user_ids) > 0:
                        discussion_type = "TASK_HUMAN_HUMAN_DISCUSSION"
                        
                    comment_thread.insert(0, discussion_type)
                    comment_threads_long.append(comment_thread)
                else: 
                    comment_thread.insert(0, discussion_type)
                    comment_threads_short.append(comment_thread)   
        
            else:
                print(chat["type"])        
                
        # update total amount of replies
        total_replies += n
        # average length of conversation per document        
        doc_stat[row["_id"]] = n / len(info)
        
        comment_threads_long_df = pd.DataFrame(data = comment_threads_long)
        comment_threads_long_df.transpose().to_csv(os.path.join('output', 'comments', 'comment_threads_long.tsv'), sep="\t")
        
        comment_threads_short_df = pd.DataFrame(data = comment_threads_short)
        comment_threads_short_df.transpose().to_csv(os.path.join('output', 'comments', 'comment_threads_short.tsv'), sep="\t")

        
print("Total chats: (incl. new & cancelled, i.e. without text)", total_chats)
#print("Total chats with before excluding ai functions (legacy count incl. extend, summarize, translate):", total_chats_before)

print("Total replies:", total_replies)
print("max stat:", max_stat)
print("Avg reply per chat", total_replies / total_chats)
print("Avg Comments per document", sum(doc_stat.values()) / len(doc_stat))

print("Like status:", like_status)
print("Dislike status:", dislike_status)

print("Comments without text, e.g. new or cancelled", comments_without_text)
print("Comments with text, e.g. posted or approved; includes also tasks that were run by users (play button)", total_chats - comments_without_text)

print("User created comments", user_created_comments)
print("AI-Human comment: AI created comments", ai_created_comments)
print("Comments total: user created + AI created:", user_created_comments + ai_created_comments)

print("Task comment count:", task_comment_count)
print("Task comment by users:", task_comment_by_users_count)
print("Task comment by ai:", task_comment_by_ai)
print("Comments per task:")
display(pd.Series(task_id_list).value_counts().describe())

print("Human-AI comment: Mentions in initial comment:", ai_start_text_mentions)
print("Human-AI replies: Mentions in replies:", ai_mentions_replies)
print("Total ai mentions (incl. mentions in replies)", ai_mentions)

print("User replies", user_replies)
print("AI replies", ai_replies)

print("Only a comment: ", only_a_comment)
print("Human-Human discussions:", human_human_discussions)
print("Human-AI discussions (single-turn):", human_ai_discussions)
print("Human-AI negotiation discussions (multi-turn):", human_ai_nego_discussions)
print("Human-Team-AI discussions, also other humans replied:", human_team_ai_discussions)
print("AI-Human discussions:", ai_human_discussions)

print("User input count (comment + replies - user initiated task comments)", user_input_count)

#display(user_created_comments_list)


2025-03-10 21:26:27.221570 None
2025-03-10 19:10:31.306000 None


{'typing': [],
 'mode': 'comment',
 'data': {'date': 1741633831306,
  'user': {'tag': '@bt716923',
   'id': '67cc9f69285fac3f07bcf5c8',
   'type': 'you',
   'name': 'bt716923',
   'author_id': '',
   'online': True,
   'picture': '/collab-editor/user2.svg'},
  'text': '<span class="tag-mention">@aiauthor</span>&nbsp;What are some of the names of these algorithms?'},
 'history': [{'state': 'NEW', 'time': 1741633831306},
  {'time': 1741633903124, 'state': 'POSTED'}],
 'type': 'comment',
 'state': 'POSTED',
 'suggestion': None,
 'replies': [{'replies': [],
   'id': 'UGeX9iXA0RShE9yvrwKoj',
   'history': [{'time': 1741633910721, 'state': 'POSTED'}],
   'data': {'user': {'picture': '/collab-editor/agent2.svg',
     'id': 'ai-author',
     'type': 'ai',
     'tag': '@aiauthor',
     'author_id': '',
     'online': False,
     'name': 'AI author'},
    'ai': {'data': {'prediction': 'There are several algorithms and technologies being used in the financial sector for various AI-driven tasks. H

AI_EXTEND
2025-03-10 14:45:32.920731 None
2025-03-10 13:27:31.274661 None
2025-03-10 21:26:37.271723 None
2025-03-15 07:45:45.056470 None
2025-03-10 12:55:24.853369 None
2025-03-10 14:57:38.355977 None
2025-03-10 14:57:38.355977 None
2025-03-10 14:45:32.920731 None
2025-03-10 12:55:24.853369 None
2025-03-10 14:45:04.307019 None
2025-03-09 22:19:04.312097 None
2025-03-10 15:02:33.658804 None
2025-03-09 21:16:08.194000 2025-03-10 14:44:50.308000
2025-03-10 18:57:24.918045 None
2025-03-10 19:17:01.328000 None


{'history': [{'state': 'NEW', 'time': 1741634221328},
  {'state': 'POSTED', 'time': 1741634289603}],
 'data': {'text': '<span class="tag-mention">@aiauthor</span>&nbsp;Give me more examples of these GenAI models.&nbsp;',
  'date': 1741634221328,
  'user': {'id': '67cc9f69285fac3f07bcf5c8',
   'author_id': '',
   'tag': '@bt716923',
   'type': 'you',
   'name': 'bt716923',
   'online': True,
   'picture': '/collab-editor/user2.svg'}},
 'id': 'VAatOu1QDqAt07vy7Ne8h',
 'state': 'POSTED',
 'mode': 'comment',
 'type': 'comment',
 'suggestion': None,
 'typing': [],
 'replies': [{'state': 'POSTED',
   'data': {'ai': {'data': {'prediction': "Sure, besides SORA and DALLE, here are some more examples of generative AI models:\n\n1. GPT-3: Developed by OpenAI, it's a language model that uses machine learning to produce human-like text. It can be used to create crafted narratives, generate code, translate languages, and much more.\n\n2. VQ-VAE-2: This model by OpenAI is capable of generating high-q

2025-03-10 14:57:38.355977 None
2025-03-10 19:19:46.784000 None


...

2025-08-14 18:21:48.475236 None
2025-08-14 18:10:08.723000 None
2025-08-14 18:15:14.572000 None


{'state': 'APPROVED',
 'data': {'user': {'tag': '@joy_angel',
   'id': '689e22d5e90ffc190a96c2e0',
   'type': 'you',
   'name': 'joy_angel',
   'online': True,
   'author_id': '',
   'picture': '/collab-editor/user2.svg'},
  'date': 1755195314572,
  'text': '<span class="tag-mention">@aiGrammar</span>&nbsp;can you review the grammar and sentence structure'},
 'suggestion': {'id': 'Upd2BrNUoO9kSnuLsrSl2', 'user': 'Grammar'},
 'replies': [{'data': {'ai': {'data': {'prediction': 'The revised text you\'ve provided is quite clear and well-structured. However, here are a few additional suggestions to enhance its clarity and flow:\n\n---\n\n**Revised Version:**\n\nWe were selected among various female applicants to be part of this Scholarship program. As a kick-off event, we were invited to the Deutsche Telekom headquarters in Bonn, where the Welcome Event took place. It was a very insightful day; we were able to listen to the experiences of other scholars and also meet female leaders at Deut

Total chats: (incl. new & cancelled, i.e. without text) 477
Total replies: 543
max stat: {'67ce983b285fac3f07bcf5e8': 7, '67c97527285fac3f07bcf587': 1, '67c88f88285fac3f07bcf585': 7, '67d43b390b9539e69a3a86b9': 5, '67d5d8160b9539e69a3a86d8': 8, '67d08e87285fac3f07bcf64b': 1, '68833fd687cf5fec60dc801d': 4, '6879e8fd87cf5fec60dc7fe1': 3, '688364bf87cf5fec60dc8026': 6, '6877c26e87cf5fec60dc7fda': 7, '6889e57087cf5fec60dc8092': 1, '688c6add87cf5fec60dc80ef': 1, '68907a6e87cf5fec60dc8107': 0, '6896071ee90ffc190a96c285': 3}
Avg reply per chat 1.1383647798742138
Avg Comments per document 0.9820576699578578
Like status: 34
Dislike status: 12
Comments without text, e.g. new or cancelled 37
Comments with text, e.g. posted or approved; includes also tasks that were run by users (play button) 440
User created comments 83
AI-Human comment: AI created comments 324
Comments total: user created + AI created: 407
Task comment count: 357
Task comment by users: 33
Task comment by ai: 324
Comments per tas

count    53.000000
mean      6.735849
std       7.419182
min       1.000000
25%       2.000000
50%       4.000000
75%       7.000000
max      37.000000
Name: count, dtype: float64

Human-AI comment: Mentions in initial comment: 68
Human-AI replies: Mentions in replies: 36
Total ai mentions (incl. mentions in replies) 104
User replies 64
AI replies 477
Only a comment:  5
Human-Human discussions: 3
Human-AI discussions (single-turn): 56
Human-AI negotiation discussions (multi-turn): 12
Human-Team-AI discussions, also other humans replied: 6
AI-Human discussions: 6
User input count (comment + replies - user initiated task comments) 147


In [59]:
# ad hoc comment tasks
display(ad_hoc_comment_tasks)
display(len(ad_hoc_comment_tasks))

for aht in ad_hoc_comment_tasks:
    aht["text"] = BeautifulSoup(str(aht["text"]), 'lxml').get_text()
    aht["group_id"] = user_lookup[aht["user_id"]][0]
    aht["user"] = user_lookup[aht["user_id"]][1]
    
ad_hoc_comment_tasks_df = pd.DataFrame(data = ad_hoc_comment_tasks, columns = ['text', 'group_id', 'user', 'user_id', 'comment_id', "assignee_id", "assignee_name"])
display(ad_hoc_comment_tasks_df)

display(ad_hoc_comment_tasks_df["assignee_id"].value_counts())
display(ad_hoc_comment_tasks_df["assignee_id"].value_counts().describe())

ad_hoc_comment_tasks_df.to_csv(os.path.join('output', 'comments', 'ad_hoc_task_comments.tsv'), sep="\t")

[{'text': '<span class="tag-mention">@aiauthor</span>&nbsp;What are some of the names of these algorithms?',
  'user_id': '67cc9f69285fac3f07bcf5c8',
  'comment_id': '7DTFbKN6yVngm0vX_3FoB',
  'assignee_id': 'ai-author',
  'assignee_name': 'AI author'},
 {'text': '<span class="tag-mention">@aiauthor</span>&nbsp;Give me more examples of these GenAI models.&nbsp;',
  'user_id': '67cc9f69285fac3f07bcf5c8',
  'comment_id': 'VAatOu1QDqAt07vy7Ne8h',
  'assignee_id': 'ai-author',
  'assignee_name': 'AI author'},
 {'text': '<span class="tag-mention">@aiAI Max</span>&nbsp;what\'s the history of AI ?',
  'user_id': '67cc9f69285fac3f07bcf5c8',
  'comment_id': '6daYdxwEEcG58TSTMV6aw',
  'assignee_id': '67cdfef4285fac3f07bcf5d3',
  'assignee_name': 'AI Max'},
 {'text': '<span class="tag-mention">@aiauthor</span>&nbsp;Please summarize this part of the text.',
  'user_id': '67cc9f69285fac3f07bcf5c8',
  'comment_id': 'CiDt21N2QEP1m7Le0k_mj',
  'assignee_id': 'ai-author',
  'assignee_name': 'AI author'

68

,text,group_id,user,user_id,comment_id,assignee_id,assignee_name
0,@aiauthor What are some of the names of these ...,3,1378,67cc9f69285fac3f07bcf5c8,7DTFbKN6yVngm0vX_3FoB,ai-author,AI author
1,@aiauthor Give me more examples of these GenAI...,3,1378,67cc9f69285fac3f07bcf5c8,VAatOu1QDqAt07vy7Ne8h,ai-author,AI author
2,@aiAI Max what's the history of AI ?,3,1378,67cc9f69285fac3f07bcf5c8,6daYdxwEEcG58TSTMV6aw,67cdfef4285fac3f07bcf5d3,AI Max
3,@aiauthor Please summarize this part of the text.,3,1378,67cc9f69285fac3f07bcf5c8,CiDt21N2QEP1m7Le0k_mj,ai-author,AI author
4,"@aiErika , could you please make this text sho...",3,1374,67cdfa30285fac3f07bcf5cc,SuRM3VccpleBCCxUJqLEG,67cdfc85285fac3f07bcf5cf,Erika
...,...,...,...,...,...,...,...
63,@aiSynthesizer read the papers listed here and...,11,2463,6889e71f87cf5fec60dc8094,5R0Qw-8ibtsl200781iq7,68931d9887cf5fec60dc810f,Synthesizer
64,@aiGrammar kindly review the paragraph for spe...,14,2471,689e22d5e90ffc190a96c2e0,-49Y2A8HTMs4Vy6iEEGEr,689e2470e90ffc190a96c2e4,Grammar
65,@aiFact Checker is this accurate?,14,2471,689e22d5e90ffc190a96c2e0,-6VoVK0uNQTmvgYAqZeQP,689e24b5e90ffc190a96c2e6,Fact Checker
66,@aiFact Checker is this accurate?,14,2471,689e22d5e90ffc190a96c2e0,ciFQV8ZgM6nhN8nzzNk0D,689e24b5e90ffc190a96c2e6,Fact Checker


assignee_id
6887772287cf5fec60dc8036    16
ai-author                   14
688b7f6b87cf5fec60dc80b7     8
688bdddd87cf5fec60dc80d6     6
67e5cb6450d39f32454bf2fa     3
67d445e30b9539e69a3a86c3     3
67d445d10b9539e69a3a86c1     3
6888b66487cf5fec60dc806d     3
67cdfc85285fac3f07bcf5cf     2
68931d9887cf5fec60dc810f     2
689e2470e90ffc190a96c2e4     2
689e24b5e90ffc190a96c2e6     2
67cdfef4285fac3f07bcf5d3     1
67d83558edfe680c1b0f165e     1
6880e49587cf5fec60dc8009     1
688a6a2087cf5fec60dc80a2     1
Name: count, dtype: int64

count    16.00000
mean      4.25000
std       4.61158
min       1.00000
25%       1.75000
50%       2.50000
75%       3.75000
max      16.00000
Name: count, dtype: float64

In [60]:
# ad hoc comment tasks
display(human_ai_discussions_list)
display(len(human_ai_discussions_list))

for had in human_ai_discussions_list:
    had["text"] = BeautifulSoup(str(had["text"]), 'lxml').get_text()
    had["group_id"] = user_lookup[had["user_id"]][0]
    had["user"] = user_lookup[had["user_id"]][1]
    
human_ai_discussions_df = pd.DataFrame(data = human_ai_discussions_list, columns = ['text', 'group_id', 'user', 'user_id', 'comment_id', "assignee_id"])
display(human_ai_discussions_df)
human_ai_discussions_df.to_csv(os.path.join('output', 'comments', 'human_ai_discussions.tsv'), sep="\t")

[{'chat': {'typing': [],
   'mode': 'comment',
   'data': {'date': 1741633831306,
    'user': {'tag': '@bt716923',
     'id': '67cc9f69285fac3f07bcf5c8',
     'type': 'you',
     'name': 'bt716923',
     'author_id': '',
     'online': True,
     'picture': '/collab-editor/user2.svg'},
    'text': '<span class="tag-mention">@aiauthor</span>&nbsp;What are some of the names of these algorithms?'},
   'history': [{'state': 'NEW', 'time': 1741633831306},
    {'time': 1741633903124, 'state': 'POSTED'}],
   'type': 'comment',
   'state': 'POSTED',
   'suggestion': None,
   'replies': [{'replies': [],
     'id': 'UGeX9iXA0RShE9yvrwKoj',
     'history': [{'time': 1741633910721, 'state': 'POSTED'}],
     'data': {'user': {'picture': '/collab-editor/agent2.svg',
       'id': 'ai-author',
       'type': 'ai',
       'tag': '@aiauthor',
       'author_id': '',
       'online': False,
       'name': 'AI author'},
      'ai': {'data': {'prediction': 'There are several algorithms and technologies bei

56

,text,group_id,user,user_id,comment_id,assignee_id
0,@aiauthor What are some of the names of these ...,3,1378,67cc9f69285fac3f07bcf5c8,7DTFbKN6yVngm0vX_3FoB,NaN
1,@aiauthor Give me more examples of these GenAI...,3,1378,67cc9f69285fac3f07bcf5c8,VAatOu1QDqAt07vy7Ne8h,NaN
2,@aiAI Max what's the history of AI ?,3,1378,67cc9f69285fac3f07bcf5c8,6daYdxwEEcG58TSTMV6aw,NaN
3,@aiauthor Please summarize this part of the text.,3,1378,67cc9f69285fac3f07bcf5c8,CiDt21N2QEP1m7Le0k_mj,NaN
4,"@aiErika , could you please make this text sho...",3,1374,67cdfa30285fac3f07bcf5cc,SuRM3VccpleBCCxUJqLEG,NaN
5,"@aiErika, could you please summarize the writt...",3,1374,67cdfa30285fac3f07bcf5cc,gobhqckqLD3t1FGuJG6l8,NaN
6,"@aiauthor , Write the content for this intro e...",1,1380,67ce9cb4285fac3f07bcf5ec,MfaSsmCS8az5tJiEdtIX1,NaN
7,"@aiauthor , Enhance professional tone",1,1380,67ce9cb4285fac3f07bcf5ec,6PoKqRSqm5nBKdEJnF7KG,NaN
8,@aiauthor can you give any statisctics about ...,1,1379,67ce982f285fac3f07bcf5e7,nOqZNstq0_qlc9q0-0c4X,NaN
9,"@aiauthor Fix my text, enhance professional to...",1,1380,67ce9cb4285fac3f07bcf5ec,mKJ7hgCmK8cXI1G8P2fjG,NaN


In [61]:
# ad hoc reply tasks
display(ad_hoc_reply_tasks)
display(len(ad_hoc_reply_tasks))

for aht in ad_hoc_reply_tasks:
    aht["text"] = BeautifulSoup(str(aht["text"]), 'lxml').get_text()
    aht["group_id"] = user_lookup[aht["user_id"]][0]
    aht["user"] = user_lookup[aht["user_id"]][1]
    
ad_hoc_reply_tasks_df = pd.DataFrame(data = ad_hoc_reply_tasks, columns = ['text', 'group_id', 'user', 'user_id', 'comment_id', 'assignee_id', 'assignee_name'])
display(ad_hoc_reply_tasks_df)
ad_hoc_reply_tasks_df.to_csv(os.path.join('output', 'comments', 'ad_hoc_task_replies.tsv'), sep="\t")

[{'text': '<span class="tag-mention">@aiAI Max</span> <br>Could you please make your answer shorter so that I can put it in the text.',
  'user_id': '67cdfa30285fac3f07bcf5cc',
  'comment_id': 'XzJllbkbnRCR-EZRbYc37',
  'user_name': '67cdfa30285fac3f07bcf5cc',
  'assignee_id': '67cdfef4285fac3f07bcf5d3',
  'assignee_name': 'AI Max'},
 {'text': '<div><span class="tag-mention">@aiAI Max</span> , yes I do. Give me some papers and build them in the essay</div>',
  'user_id': '67cdfa30285fac3f07bcf5cc',
  'comment_id': 'XzJllbkbnRCR-EZRbYc37',
  'user_name': '67cdfa30285fac3f07bcf5cc',
  'assignee_id': '67cdfef4285fac3f07bcf5d3',
  'assignee_name': 'AI Max'},
 {'text': '<div><span class="tag-mention">@aiAI Max</span>, ok, make it so that I can put it in the text</div>',
  'user_id': '67cdfa30285fac3f07bcf5cc',
  'comment_id': 'XzJllbkbnRCR-EZRbYc37',
  'user_name': '67cdfa30285fac3f07bcf5cc',
  'assignee_id': '67cdfef4285fac3f07bcf5d3',
  'assignee_name': 'AI Max'},
 {'text': '<div><span cl

36

,text,group_id,user,user_id,comment_id,assignee_id,assignee_name
0,@aiAI Max Could you please make your answer sh...,3,1374,67cdfa30285fac3f07bcf5cc,XzJllbkbnRCR-EZRbYc37,67cdfef4285fac3f07bcf5d3,AI Max
1,"@aiAI Max , yes I do. Give me some papers and ...",3,1374,67cdfa30285fac3f07bcf5cc,XzJllbkbnRCR-EZRbYc37,67cdfef4285fac3f07bcf5d3,AI Max
2,"@aiAI Max, ok, make it so that I can put it in...",3,1374,67cdfa30285fac3f07bcf5cc,XzJllbkbnRCR-EZRbYc37,67cdfef4285fac3f07bcf5d3,AI Max
3,"@aiauthor , could you please make your respons...",3,1374,67cdfa30285fac3f07bcf5cc,JdQKwfkzfixwA-bBM8KrJ,ai-author,AI author
4,"I added @aiBob , to give full analysis of the ...",1,1380,67ce9cb4285fac3f07bcf5ec,2TPbqrhSqTafGZwjVQFi4,67cea551285fac3f07bcf604,Bob
5,@aiauthor give us topics ideas for an essay,1,1379,67ce982f285fac3f07bcf5e7,XQt5XrkOapXdkdaV-J3Cn,ai-author,AI author
6,"@aiauthor give us the structure of the essay ""...",1,1379,67ce982f285fac3f07bcf5e7,XQt5XrkOapXdkdaV-J3Cn,ai-author,AI author
7,@aiauthor humanize the text,1,1379,67ce982f285fac3f07bcf5e7,18-cROhkXZwVmH9qkyXtx,ai-author,AI author
8,@aiauthor Make the text more enaging,1,1379,67ce982f285fac3f07bcf5e7,18-cROhkXZwVmH9qkyXtx,ai-author,AI author
9,@aiBest HCI Professor Provide the best fitting...,4,1383,67bc761a7a670b68746905a0,XUnGafYeCxruxeHPC3V5N,67d445e30b9539e69a3a86c3,Best HCI Professor


[P]Average number of comments per group

In [62]:
# user 67d5bdff0b9539e69a3a86c9 and 67d5d8120b9539e69a3a86d7 appear to be the same person?
# skip 67d5bdff0b9539e69a3a86c9 since it is a second account and was only used after the IV

user_created_comment_times = []
ghost_comments = []
for comment in user_created_comments_list:       
    if comment['data']['user']['id'] == "67d5bdff0b9539e69a3a86c9":
        ghost_comments.append({"date": datetime.utcfromtimestamp(comment['data']['date'] / 1000), "comment": comment})
        continue
    date = datetime.utcfromtimestamp(comment['data']['date'] / 1000)
    group_id = user_id = user_lookup[comment['data']['user']['id']][0]
    user_id = user_lookup[comment['data']['user']['id']][1]
    entry = {
        'id': comment['id'],
        'date': date,
        'group_id': group_id,
        'user_id': user_id,
        'comment': comment
    }
    user_created_comment_times.append(entry)
    
user_created_comments_df = pd.DataFrame(data = user_created_comment_times, columns = ['id', 'date', 'group_id', 'user_id', 'comment'])

display(user_created_comments_df)
display(ghost_comments)

# 2, 5, 12 missing?
display(user_created_comments_df["group_id"].value_counts())
display(user_created_comments_df["group_id"].value_counts().describe())

,id,date,group_id,user_id,comment
0,7DTFbKN6yVngm0vX_3FoB,2025-03-10 19:10:31.306,3,1378,"{'typing': [], 'mode': 'comment', 'data': {'da..."
1,VAatOu1QDqAt07vy7Ne8h,2025-03-10 19:17:01.328,3,1378,"{'history': [{'state': 'NEW', 'time': 17416342..."
2,6daYdxwEEcG58TSTMV6aw,2025-03-10 19:19:46.784,3,1378,{'replies': [{'data': {'ai': {'data': {'predic...
3,CiDt21N2QEP1m7Le0k_mj,2025-03-10 19:23:15.858,3,1378,"{'state': 'POSTED', 'type': 'comment', 'id': '..."
4,1FA70-gC_UGS4rYtQpsEK,2025-03-10 19:40:49.558,3,1378,"{'history': [{'state': 'NEW', 'time': 17416356..."
...,...,...,...,...,...
78,5R0Qw-8ibtsl200781iq7,2025-08-06 09:19:21.451,11,2463,"{'suggestion': {'id': '33rSOoMIJV3L8ZzVqNkj6',..."
79,-49Y2A8HTMs4Vy6iEEGEr,2025-08-14 18:13:35.374,14,2471,"{'state': 'APPROVED', 'mode': 'comment', 'data..."
80,-6VoVK0uNQTmvgYAqZeQP,2025-08-14 18:12:34.161,14,2471,"{'suggestion': None, 'typing': [], 'mode': 'co..."
81,ciFQV8ZgM6nhN8nzzNk0D,2025-08-14 18:10:35.672,14,2471,"{'suggestion': None, 'state': 'APPROVED', 'typ..."


[]

group_id
7     22
9     18
1     14
4     11
3      7
14     4
11     3
8      2
6      1
10     1
Name: count, dtype: int64

count    10.000000
mean      8.300000
std       7.572611
min       1.000000
25%       2.250000
50%       5.500000
75%      13.250000
max      22.000000
Name: count, dtype: float64

In [63]:
# mentions in replies
print("Mentions in replies as a fraction of user input:", ai_mentions_replies / user_input_count)
print("Percentage of AI mentions in replies:", ai_mentions_replies / ai_mentions * 100)

Mentions in replies as a fraction of user input: 0.24489795918367346
Percentage of AI mentions in replies: 34.61538461538461


In [64]:
finished_df = finished_participants[
    finished_participants["event"] == "DOCUMENT_TEXT_PASTE"
]

display(finished_df["data"])

# count if pasted text was from the card
paste_count = 0

for idx, row in finished_df.iterrows():
    info = json.loads(row["data"])

    document_id = info["document_id"]

    pasted_text = info["pasted_text"]

    display(document_id)
    current_document = data[data["_id"] == document_id].iloc[0]

    try:
        info = json.loads(current_document["cards"])

        for chat in info.values():

            
            if "ai" in chat and "prediction" in chat["ai"]:
                if pasted_text in chat["ai"]["prediction"]:
                    paste_count += 1
                    card_ids.add(chat["id"])
                    # print(pasted_text, chat["ai"]["prediction"])
                    continue

            if "replies" in chat:
                for reply in chat["replies"]:
                    # print(reply)
                    if "data" in reply and "ai" in reply["data"] and "data" in reply["data"]["ai"] and "prediction" in reply["data"]["ai"]["data"]:
                        if pasted_text in reply["data"]["ai"]["data"]["prediction"]:
                            paste_count += 1
                            card_ids.add(chat["id"])
                            continue
        
    except Exception:
        continue

print("Text copied and pasted from cards (w/o group 6): ", paste_count, "total cards", len(card_ids))

106     {"user_id": "67cb6623285fac3f07bcf59a", "usern...
167     {"user_id": "67cb6627285fac3f07bcf59b", "usern...
416     {"user_id": "67cdfa30285fac3f07bcf5cc", "usern...
427     {"user_id": "67cdfa30285fac3f07bcf5cc", "usern...
434     {"user_id": "67cdfa30285fac3f07bcf5cc", "usern...
                              ...                        
6480    {"user_id": "68a75365ecc07d2944852275", "usern...
6495    {"user_id": "68a75365ecc07d2944852275", "usern...
6550    {"user_id": "68a75365ecc07d2944852275", "usern...
6558    {"user_id": "68a75365ecc07d2944852275", "usern...
6564    {"user_id": "68a75365ecc07d2944852275", "usern...
Name: data, Length: 117, dtype: object

'67c97527285fac3f07bcf587'

'67c97527285fac3f07bcf587'

'67c88f88285fac3f07bcf585'

'67c88f88285fac3f07bcf585'

...

'6889e57087cf5fec60dc8092'

'6889e57087cf5fec60dc8092'

'6889e57087cf5fec60dc8092'

'6889e57087cf5fec60dc8092'

Text copied and pasted from cards (w/o group 6):  90 total cards 163


In [65]:
# count APPROVED cards for group 4 because they don't have log data
six_group = data[data["_id"] == "67d43b390b9539e69a3a86b9"]

count_approves = 0

for idx, row in six_group.iterrows():
    # get cards
    try:
        info = json.loads(row["cards"])

        for chat in info.values():
            if 'state' in chat:
                if chat["state"] == "APPROVED":
                    count_approves += 1
                    card_ids.add(chat["id"])
                    print(chat["id"])
     


    except Exception:
        continue

print("Approved cards of group 6:", count_approves)
print(len(card_ids))

hJ9sUH3y-UK90rmCILf4j
fCt-Hwk_iMYOKu0iEiqSi
mLNRXhlEyYKcaLvIaxh_3
MK7nNP5gwF6Ecveyyk_7Z
Approved cards of group 6: 4
167


In [66]:
# fraction of applied suggestions
len(card_ids) / (total_chats - comments_without_text)

0.3795454545454545

In [67]:
# for the last group count if text from card is in document text (only for group 6 because they don't have log data)
six_paste_count = 0

for idx, row in six_group.iterrows():
    text = row["text"]
    try:
        info = json.loads(row["cards"])
        for chat in info.values():
            if "ai" in chat and "prediction" in chat["ai"]:
                if chat["ai"]["prediction"] in text:
                    six_paste_count += 1
                    card_ids.add(chat["id"])

                    continue

            if "replies" in chat:
                for reply in chat["replies"]:
                    if "data" in reply and "ai" in reply["data"] and "data" in reply["data"]["ai"] and "prediction" in reply["data"]["ai"]["data"]:
                        if reply["data"]["ai"]["data"]["prediction"] in text:
                            six_paste_count += 1
                            card_ids.add(chat["id"])
                            continue
        
        


    except Exception:
        print(document_id)

        continue

print(six_paste_count)
print(len(card_ids))

0
167


Timeline Analysis

In [68]:
def consecutive_focus_event_rows(event_df, fieldname, bypass = []):
    consecutive_rows_list = []
    for n, g in event_df.groupby(['participant_id', (event_df[fieldname] != event_df[fieldname].shift()).cumsum()]):
        code_list = g[fieldname].tolist()
        
        if code_list[0] in bypass:
            for index, row in g.iterrows():
                
                consecutive_rows_list.append({
                    'participant_id': row['participant_id'],
                    fieldname: code_list[0],
                    'participant_token': row['participant_token'],
                    'consecutive_rows': [code_list[0]],
                    'length': -1,
                    'from_id': row['id'],
                    'to_id': row['id'],
                    'from_time': row['time'],
                    'to_time': row['time']
                   # 'state': state
                })
            
        else: 
            consecutive_rows_list.append({
                'participant_id': n[0],
                fieldname: code_list[0],
                'participant_token': g['participant_token'].iloc[0],
                'consecutive_rows': code_list,
                'length': len(code_list),
                'from_id': g['id'].iloc[0],
                'to_id': g['id'].iloc[-1],
                'from_time': g['time'].iloc[0],
                'to_time': g['time'].iloc[-1]
               # 'state': state
            })

    consecutive_rows_columns = ['participant_id', fieldname, 'participant_token', 'consecutive_rows', 'length', 'from_id', 'to_id', 'from_time', 'to_time']
    return pd.DataFrame(data = consecutive_rows_list, columns = consecutive_rows_columns)

In [69]:
def prepare_events_for_timeline(event_df, fieldname):
    rows = []
    for index, row in event_df.iterrows():
        rows.append({
            'participant_id': row['participant_id'],
            fieldname: row[fieldname],
            'participant_token': row['participant_token'],
            'id': row['id'],
            'time': row['time']
        })
        
    columns = ['participant_id', fieldname, 'participant_token','id', 'time']
    return pd.DataFrame(data = rows, columns = columns)

In [70]:
focus_events_df = finished_participants[finished_participants['event'].isin(["DOCUMENT_VIEW", 'FINAL_PROCEED_TEXT', 'COMMENT_ADD', 'COMMENT_TASK_ADD', 'COMMENT_POST', 'REPLY_POST', 'COMMENT_DELETE', 'COMMENT_CANCEL', 'COMMENT_EDIT', 'COMMENT_EDIT_CANCEL', 'COMMENT_APPROVE', 'COMMENT_REPLY_APPROVE','REPLY_START', 'REPLY_EDIT_TOGGLE', 'REPLY_EDIT', 'REPLY_EDIT_CANCEL', 'REPLY_DELETE', 'REPLY_CANCEL', 'REPLY_AI', 'SUGGESTION_INSERT_AFTER', 'SUGGESTION_TAKE_OVER', 'SUGGESTION_COPY_TO_CLIPBOARD', 'SUGGESTION_VIEWDETAIL', 'SUGGESTION_CLOSEDETAIL'])]

consecutive_focus_events_df = consecutive_focus_event_rows(focus_events_df, 'event', ['COMMENT_ADD', 'COMMENT_TASK_ADD', 'COMMENT_POST', 'REPLY_POST', 'COMMENT_DELETE', 'COMMENT_CANCEL', 'COMMENT_EDIT', 'COMMENT_EDIT_CANCEL', 'COMMENT_APPROVE', 'COMMENT_REPLY_APPROVE','REPLY_START', 'REPLY_EDIT_TOGGLE', 'REPLY_EDIT', 'REPLY_EDIT_CANCEL', 'REPLY_DELETE', 'REPLY_CANCEL', 'REPLY_AI', 'SUGGESTION_INSERT_AFTER', 'SUGGESTION_TAKE_OVER', 'SUGGESTION_COPY_TO_CLIPBOARD', 'SUGGESTION_VIEWDETAIL', 'SUGGESTION_CLOSEDETAIL'])
prepared_focus_events_df = prepare_events_for_timeline(focus_events_df, 'event')

# transform times to datetime
consecutive_focus_events_df = transform_time(consecutive_focus_events_df, 'from_time')
consecutive_focus_events_df = transform_time(consecutive_focus_events_df, 'to_time')

prepared_focus_events_df = transform_time(prepared_focus_events_df, 'time')



display(consecutive_focus_events_df[consecutive_focus_events_df["participant_id"] == 1374].head(25))
display(prepared_focus_events_df[prepared_focus_events_df["participant_id"] == 1374].head(25))

,participant_id,event,participant_token,consecutive_rows,length,from_id,to_id,from_time,to_time
0,1374,DOCUMENT_VIEW,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,"[DOCUMENT_VIEW, DOCUMENT_VIEW, DOCUMENT_VIEW, ...",5,276430.0,276518.0,2025-03-09 20:29:39.476,2025-03-09 20:50:12.536
1,1374,COMMENT_TASK_ADD,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,[COMMENT_TASK_ADD],-1,276544.0,276544.0,2025-03-09 21:16:08.194,2025-03-09 21:16:08.194
2,1374,REPLY_AI,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,[REPLY_AI],-1,276548.0,276548.0,2025-03-09 21:16:15.493,2025-03-09 21:16:15.493
3,1374,SUGGESTION_VIEWDETAIL,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,[SUGGESTION_VIEWDETAIL],-1,276551.0,276551.0,2025-03-09 21:16:19.034,2025-03-09 21:16:19.034
4,1374,REPLY_START,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,[REPLY_START],-1,276552.0,276552.0,2025-03-09 21:17:25.748,2025-03-09 21:17:25.748
5,1374,REPLY_POST,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,[REPLY_POST],-1,276557.0,276557.0,2025-03-09 21:18:20.550,2025-03-09 21:18:20.550
6,1374,REPLY_AI,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,[REPLY_AI],-1,276561.0,276561.0,2025-03-09 21:18:22.747,2025-03-09 21:18:22.747
7,1374,SUGGESTION_VIEWDETAIL,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,[SUGGESTION_VIEWDETAIL],-1,276565.0,276565.0,2025-03-09 21:18:28.022,2025-03-09 21:18:28.022
8,1374,REPLY_START,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,[REPLY_START],-1,276566.0,276566.0,2025-03-09 21:18:47.942,2025-03-09 21:18:47.942
9,1374,REPLY_START,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,[REPLY_START],-1,276569.0,276569.0,2025-03-09 21:19:33.235,2025-03-09 21:19:33.235


,participant_id,event,participant_token,id,time
29,1374,DOCUMENT_VIEW,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,276430.0,2025-03-09 20:29:39.476
30,1374,DOCUMENT_VIEW,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,276448.0,2025-03-09 20:31:05.669
31,1374,DOCUMENT_VIEW,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,276484.0,2025-03-09 20:39:52.960
32,1374,DOCUMENT_VIEW,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,276496.0,2025-03-09 20:47:50.991
33,1374,DOCUMENT_VIEW,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,276518.0,2025-03-09 20:50:12.536
34,1374,COMMENT_TASK_ADD,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,276544.0,2025-03-09 21:16:08.194
35,1374,REPLY_AI,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,276548.0,2025-03-09 21:16:15.493
36,1374,SUGGESTION_VIEWDETAIL,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,276551.0,2025-03-09 21:16:19.034
37,1374,REPLY_START,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,276552.0,2025-03-09 21:17:25.748
38,1374,REPLY_POST,93f70a37-e7d5-4bcf-adbc-d106ca47c27b,276557.0,2025-03-09 21:18:20.550


In [71]:
# participant_mapping = {
#     1379: 'G1P1',
#     1380: 'G1P2',
#     1376: 'G2P1',
#     1377: 'G2P2', 
#     1378: 'G3P1', 
#     1374: 'G3P2', 
#     1388: 'G4P1',
#     1389: 'G4P2',
#     1387: 'G5P1',
#     1390: 'G5P2'
# }

# explore events that happened throughout the final interview
#for n, g in consecutive_focus_events_df.groupby(['participant_id']):
#    print("Participant", n[0])
#    display(g.tail(50))

In [72]:
# remove events that happened at the interview
#participant_id, time

clean_consecutive_focus_events_df = remove_events_while_interviewing(consecutive_focus_events_df, remove_events_starting_from)
participant_focus_events = clean_consecutive_focus_events_df.groupby(['participant_id'])

print(participant_focus_events)

In [73]:
# get min max for each group

# Important: we consider the cleaned consecutive focus events, these events do not contain the events that happened in the final interviews.
# The windows of times (min - max) are thus shorter.

minmaxtimes_participants = {}
for pg in participants_groups:
    pg_events = clean_consecutive_focus_events_df[clean_consecutive_focus_events_df['participant_id'].isin(participants_groups[pg]["users"])]
    #display(pg_events)
    min_time = pg_events['from_time'].min()
    max_time = pg_events['to_time'].max()

    for p in participants_groups[pg]["users"]:
        minmaxtimes_participants[p] = {"min": min_time, "max": max_time}

display(minmaxtimes_participants)

{1379: {'min': Timestamp('2025-03-10 07:43:55.548000'),
  'max': Timestamp('2025-03-10 08:42:46.563000')},
 1380: {'min': Timestamp('2025-03-10 07:43:55.548000'),
  'max': Timestamp('2025-03-10 08:42:46.563000')},
 1376: {'min': Timestamp('2025-03-07 21:33:49.808000'),
  'max': Timestamp('2025-03-07 21:50:55.943000')},
 1377: {'min': Timestamp('2025-03-07 21:33:49.808000'),
  'max': Timestamp('2025-03-07 21:50:55.943000')},
 1378: {'min': Timestamp('2025-03-08 19:52:11.143000'),
  'max': Timestamp('2025-03-15 07:45:19.751000')},
 1374: {'min': Timestamp('2025-03-08 19:52:11.143000'),
  'max': Timestamp('2025-03-15 07:45:19.751000')},
 1383: {'min': Timestamp('2025-03-14 15:02:19.172000'),
  'max': Timestamp('2025-03-28 12:40:02.313000')},
 1384: {'min': Timestamp('2025-03-14 15:02:19.172000'),
  'max': Timestamp('2025-03-28 12:40:02.313000')},
 1385: {'min': Timestamp('2025-03-14 15:02:19.172000'),
  'max': Timestamp('2025-03-28 12:40:02.313000')},
 1387: {'min': Timestamp('2025-03-15 

[S] When tasks were created

In [74]:
display(user_created_comments_df)

comment_creation_time_percentages = []
for p in minmaxtimes_participants:
    min_time = minmaxtimes_participants[p]['min']
    max_time = minmaxtimes_participants[p]['max']
    session_length = (max_time - min_time).total_seconds()
 
    user_agents = user_created_comments_df[user_created_comments_df['user_id'] == p]
    for index, row in user_agents.iterrows():
        diff = (row['date'] - min_time).total_seconds()
        percentage_of_time = 100 / session_length * diff
        comment_creation_time_percentages.append([p, row['group_id'], percentage_of_time])

time_to_creation_percentage = pd.DataFrame(data=comment_creation_time_percentages, columns=['user', 'group_id', 'time_percentage'])


display(time_to_creation_percentage[time_to_creation_percentage['time_percentage'] < 25])
display(100 / len(user_created_comments_df) * len(time_to_creation_percentage[time_to_creation_percentage['time_percentage'] < 25]))
display(time_to_creation_percentage[(time_to_creation_percentage['time_percentage'] > 25) & (time_to_creation_percentage['time_percentage'] < 50)])
display(100 / len(user_created_comments_df) * len(time_to_creation_percentage[(time_to_creation_percentage['time_percentage'] > 25) & (time_to_creation_percentage['time_percentage'] < 50)]))
display(time_to_creation_percentage[(time_to_creation_percentage['time_percentage'] > 50) & (time_to_creation_percentage['time_percentage'] < 75)])
display(100 / len(user_created_comments_df) * len(time_to_creation_percentage[(time_to_creation_percentage['time_percentage'] > 50) & (time_to_creation_percentage['time_percentage'] < 75)]))
display(time_to_creation_percentage[time_to_creation_percentage['time_percentage'] > 75])
display(100 / len(user_created_comments_df) * len(time_to_creation_percentage[time_to_creation_percentage['time_percentage'] > 75]))

,id,date,group_id,user_id,comment
0,7DTFbKN6yVngm0vX_3FoB,2025-03-10 19:10:31.306,3,1378,"{'typing': [], 'mode': 'comment', 'data': {'da..."
1,VAatOu1QDqAt07vy7Ne8h,2025-03-10 19:17:01.328,3,1378,"{'history': [{'state': 'NEW', 'time': 17416342..."
2,6daYdxwEEcG58TSTMV6aw,2025-03-10 19:19:46.784,3,1378,{'replies': [{'data': {'ai': {'data': {'predic...
3,CiDt21N2QEP1m7Le0k_mj,2025-03-10 19:23:15.858,3,1378,"{'state': 'POSTED', 'type': 'comment', 'id': '..."
4,1FA70-gC_UGS4rYtQpsEK,2025-03-10 19:40:49.558,3,1378,"{'history': [{'state': 'NEW', 'time': 17416356..."
...,...,...,...,...,...
78,5R0Qw-8ibtsl200781iq7,2025-08-06 09:19:21.451,11,2463,"{'suggestion': {'id': '33rSOoMIJV3L8ZzVqNkj6',..."
79,-49Y2A8HTMs4Vy6iEEGEr,2025-08-14 18:13:35.374,14,2471,"{'state': 'APPROVED', 'mode': 'comment', 'data..."
80,-6VoVK0uNQTmvgYAqZeQP,2025-08-14 18:12:34.161,14,2471,"{'suggestion': None, 'typing': [], 'mode': 'co..."
81,ciFQV8ZgM6nhN8nzzNk0D,2025-08-14 18:10:35.672,14,2471,"{'suggestion': None, 'state': 'APPROVED', 'typ..."


,user,group_id,time_percentage
30,1385,4,0.021064
31,1385,4,0.003853
77,2463,11,0.002734


3.6144578313253017

,user,group_id,time_percentage
1,1379,1,49.440968
5,1379,1,43.286194
6,1380,1,35.349298
8,1380,1,47.246585
11,1380,1,34.886258
14,1378,3,30.346333
15,1378,3,30.415832
16,1378,3,30.445315
17,1378,3,30.482571
18,1378,3,30.670333


16.867469879518072

,user,group_id,time_percentage
0,1379,1,54.446158
2,1379,1,71.227876
3,1379,1,58.491793
7,1380,1,52.161574
10,1380,1,64.918416
13,1380,1,53.131975
63,2464,9,51.039374
64,2464,9,50.161819
80,2471,14,72.771982
81,2471,14,66.667474


12.048192771084338

,user,group_id,time_percentage
4,1379,1,90.928189
9,1380,1,87.593907
12,1380,1,83.567926
21,1384,4,95.668177
22,1384,4,95.660133
23,1384,4,95.779218
24,1384,4,95.594667
25,1384,4,95.792312
26,1384,4,95.672067
27,1384,4,95.816417


67.46987951807229